# CALE f_clue Norm Bimodality Investigation

**Primary author:** Victoria

**Builds on:**
- *g1_basic_evaluation.ipynb* (Victoria — norm histograms, T=0/T=1 computation, `rowwise_cosine` helper, embedding-loading conventions)
- *specs/cale_fclue_norm_bimodality.md* (Victoria — this notebook's section outline, outputs, and figure naming)

**Prompt engineering:** Victoria
**AI assistance:** Claude / Claude Code (Anthropic)
**Environment:** Local

The g1 basic evaluation surfaced a visible bimodality in the g_stock f_clue
(val) L2 norm distribution, with peaks near 29.5 and 31.5. This notebook
investigates the source of that bimodality through systematic subsetting,
rules out computational error, characterizes which embedding dimensions drive
the two modes, and tests whether the pattern propagates into cosine or
ATE-relevant measures.

The finding is a CALE model characterization: an idiosyncrasy of how the
pretrained model processes tagged text in cryptic clue context, analogous to
the WordNet sense-selection idiosyncrasies documented elsewhere. It does not
invalidate prior analyses but should be understood and tracked.

Reads artifacts under `data/filtered_split/wn_synset/`,
`data/embeddings/{g_stock, g1}/`, and the shared `data/id_map.csv` +
`data/puzzle_metadata.csv` (for publisher attribution). Writes numerical
results to
`outputs/cale_fclue_norm_bimodality-results.md` and figures to
`outputs/figures/fclue_bimodal_*.png`. Produces no new data artifacts.


## §0 — Setup

Imports and environment auto-detection. This notebook lives under
`custom_embedding_model/planning/exploration/`, so the component root is two
directories up and the project root is three directories up. `RANDOM_STATE`
is pinned once; all sampling and bin edges are deterministic.


In [ ]:
# === Imports and configuration
import json
import re
import time
from collections import Counter
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
from scipy.stats import spearmanr
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

# --- Environment auto-detection ---
try:
    IS_COLAB = "google.colab" in str(get_ipython())
except NameError:
    IS_COLAB = False

IS_GREATLAKES = Path("/nfs/turbo").exists()

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/Research Project - NLP CCC\'s/ccc-project")
elif IS_GREATLAKES:
    PROJECT_ROOT = Path.home() / "ccc-project"
else:
    # planning/exploration/ -> custom_embedding_model/ -> ccc-project/
    PROJECT_ROOT = Path("../../..").resolve()

COMPONENT_ROOT = PROJECT_ROOT / "custom_embedding_model"
DATA_DIR       = COMPONENT_ROOT / "data"
WN_DIR         = DATA_DIR / "filtered_split" / "wn_synset"
EMBED_DIR      = DATA_DIR / "embeddings"
OUTPUT_DIR     = COMPONENT_ROOT / "outputs"
FIG_DIR        = OUTPUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
SHARED_DATA    = PROJECT_ROOT / "data"   # id_map.csv, puzzle_metadata.csv

env_label = "Colab" if IS_COLAB else ("Great Lakes" if IS_GREATLAKES else "Local")
print(f"Environment:   {env_label}")
print(f"PROJECT_ROOT:  {PROJECT_ROOT}")
print(f"WN_DIR:        {WN_DIR}")
print(f"EMBED_DIR:     {EMBED_DIR}")
print(f"OUTPUT_DIR:    {OUTPUT_DIR}")
print(f"SHARED_DATA:   {SHARED_DATA}")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

NOTEBOOK_T0 = time.time()
MODEL_COLORS = {"g_stock": "#1f77b4", "g1": "#ff7f0e"}
MODEL_NAMES  = ["g_stock", "g1"]


In [ ]:
# === 2-component GMM fitting and overlay helper
# Reusable helper for fitting a 2-component Gaussian Mixture Model to each
# norm/cosine/L2 distribution that appears in this notebook, and overlaying
# the component PDFs on its histogram. Even visibly unimodal distributions
# get a fit so we can scan the resulting peak locations side by side: if
# the same μ values recur across many distributions, that suggests CALE has
# natural modes in its embedding-norm space rather than feature-specific
# bimodalities.
from sklearn.mixture import GaussianMixture
from scipy.stats import norm as sp_norm

def fit_gmm2(data):
    """Fit a 2-component GMM to 1D data.

    Returns a dict with keys:
        mu1, sigma1, weight1  — component with smaller mean
        mu2, sigma2, weight2  — component with larger mean
    Sorted so mu1 < mu2.
    """
    gm = GaussianMixture(n_components=2, random_state=RANDOM_STATE)
    gm.fit(data.reshape(-1, 1))
    mus     = gm.means_.ravel()
    sigmas  = np.sqrt(gm.covariances_.ravel())
    weights = gm.weights_
    # Sort by mean so the "lower" component is always (mu1, sigma1, weight1).
    order = np.argsort(mus)
    return {
        "mu1": float(mus[order[0]]),    "sigma1": float(sigmas[order[0]]), "weight1": float(weights[order[0]]),
        "mu2": float(mus[order[1]]),    "sigma2": float(sigmas[order[1]]), "weight2": float(weights[order[1]]),
    }


def overlay_gmm2(ax, data, fit, color="black", label_prefix="",
                 annotate_y_frac=0.92):
    """Overlay two Gaussian component PDFs and annotate peaks on an Axes.

    Parameters
    ----------
    ax : matplotlib Axes
        The axes with an existing density-normalized histogram.
    data : 1D array
        The raw data (used only to determine x range for the PDF curves).
    fit : dict
        Output of fit_gmm2.
    color : str
        Base color for the PDF curves and annotations.
    label_prefix : str
        Optional prefix for the legend labels (e.g. "g_stock ").
    annotate_y_frac : float
        Vertical position of peak annotations as a fraction of the current
        y-axis upper limit. Adjust if annotations overlap histogram bars.
    """
    x = np.linspace(float(data.min()), float(data.max()), 300)
    y_top = ax.get_ylim()[1]
    for i, (mu, sigma, w) in enumerate([
        (fit["mu1"], fit["sigma1"], fit["weight1"]),
        (fit["mu2"], fit["sigma2"], fit["weight2"]),
    ], start=1):
        pdf = w * sp_norm.pdf(x, loc=mu, scale=sigma)
        # Per-component PDF (dashed), then a vertical line at its peak.
        ax.plot(x, pdf, color=color, linewidth=1.5, linestyle="--",
                alpha=0.85)
        ax.axvline(mu, color=color, linewidth=1, linestyle=":",
                   alpha=0.7)
        # Stagger annotation vertically so the two μ labels don't collide
        # when the two component means are close together.
        voffset = -0.06 * y_top if i == 2 else 0
        ax.annotate(f"μ{label_prefix}={mu:.2f}",
                    xy=(mu, y_top * annotate_y_frac + voffset),
                    fontsize=8, ha="center", color=color,
                    bbox=dict(boxstyle="round,pad=0.2", fc="white",
                              ec="none", alpha=0.7))

    # Combined PDF (sum of both weighted components) as a solid line — shows
    # how well the 2-component mixture matches the overall histogram shape.
    pdf_total = (fit["weight1"] * sp_norm.pdf(x, loc=fit["mu1"], scale=fit["sigma1"])
               + fit["weight2"] * sp_norm.pdf(x, loc=fit["mu2"], scale=fit["sigma2"]))
    ax.plot(x, pdf_total, color=color, linewidth=1.8, linestyle="-",
            alpha=0.6, label=f"{label_prefix}GMM fit")


# Running collector so every cell that fits a GMM contributes a row to the
# summary table built at the end of the notebook.
gmm_fits = []


In [ ]:
# === Version reporting (Decision 18)
import sys

VERSIONS = {
    "python":     sys.version.split()[0],
    "numpy":      np.__version__,
    "pandas":     pd.__version__,
    "scipy":      scipy.__version__,
    "matplotlib": matplotlib.__version__,
    "seaborn":    sns.__version__,
}
for k, v in VERSIONS.items():
    print(f"{k:12s} {v}")


## §1 — What does embedding norm bimodality mean?

**What L2 norm represents.** The L2 norm of an embedding is its magnitude —
how far the vector sits from the origin. Under CALE, each f_clue embedding
represents a definition word contextualized by its cryptic clue. The norm
reflects how "strongly" the model activates in response to that input:
summed, squared, square-rooted contributions across all 1,024 output
dimensions.

**What two norm peaks imply.** A bimodal norm distribution means CALE
processes these inputs in two distinct regimes, producing embeddings of
systematically different magnitudes. If the two groups don't map to any
observable text property (word identity, clue length, publisher), then this
reflects internal model behavior — something about how self-attention and
the final pooling resolve for tagged spans in clue context, not something
about the text itself.

**Relationship to L2 distance and cosine similarity.** L2 distance between
two vectors depends on both their directions AND their magnitudes:
$\|a - b\|^2 = \|a\|^2 + \|b\|^2 - 2\|a\|\|b\|\cos\theta$.
So norm differences directly affect L2 distances. Cosine similarity depends
only on direction — it normalizes out the magnitude — so
$\cos\theta = a \cdot b / \|a\|\|b\|$. This implies:

- The bimodality will show up in any L2 measure involving f_clue (including
  T=1 L2 distance).
- It should **not** show up in cosine similarity, unless the two norm groups
  also differ in direction.
- T=0 measures (which use only f_wndef embeddings, not f_clue) cannot be
  affected by f_clue norm bimodality at all.

**What it means that g1 eliminates the bimodality.** g1 compressed all
f_clue norms into a tighter range (std: 1.20 → 0.75). Fine-tuning erased
CALE's two-regime behavior as part of its global compression of the
embedding space. This was not a targeted fix — g1 steamrolled all norm
variation, including the bimodal structure we investigate here.

**Relationship to T=0 cosine bimodality.** The g_stock T=0 cosine
similarity distribution also shows irregular multi-modal structure. T=0 uses
only f_wndef embeddings, so it cannot share a cause with the f_clue norm
bimodality. Any T=0 bimodality reflects the semantic properties of
definition-answer word pairs — how similar they are under g_stock's
pretrained representations — not how CALE processes clue context. These
are independent phenomena. This notebook investigates f_clue norm
bimodality only; the T=0 cosine shape is a separate question.


## §2 — Reproducing the bimodal distributions

First we confirm the bimodality is not a validation-set artifact by
comparing the full-dataset g_stock f_clue norms (239,406 rows, all splits)
against the validation-only slice (47,933 rows). Both should show the same
bimodal structure. We then reproduce the three relevant panels from the g1
basic evaluation focused on the bimodality: the f_clue norm histogram,
g_stock T=0/T=1 cosine, and g_stock T=0/T=1 L2.

The rest of the notebook proceeds on the validation slice, because that is
where both g_stock and g1 embeddings are available for comparison.


In [ ]:
# === Load clue CSVs, vocabulary, and index files
t0 = time.time()

clues_wn_filtered = pd.read_csv(
    WN_DIR / "clues_wn_filtered.csv",
    keep_default_na=False, na_values=[""],
)
clues_val = pd.read_csv(
    WN_DIR / "clues_val.csv",
    keep_default_na=False, na_values=[""],
)
vocab_wndef = pd.read_csv(
    WN_DIR / "wndef" / "vocabulary_wndef.csv",
    keep_default_na=False, na_values=[""],
)
f_clue_phrases = pd.read_csv(
    WN_DIR / "clue_phrases" / "f_clue.csv",
    keep_default_na=False, na_values=[""],
)

g1_f_clue_val_index = pd.read_csv(
    EMBED_DIR / "g1" / "f_clue_val_index.csv",
    keep_default_na=False, na_values=[""],
)
g_stock_f_clue_index = pd.read_csv(
    EMBED_DIR / "g_stock" / "f_clue_index.csv",
    keep_default_na=False, na_values=[""],
)

# Publisher attribution via the canonical join (DATA_RAW.md §4.3):
# clue_id --id_map.csv--> puzzle_id --puzzle_metadata.csv--> publisher.
# Never use clues_raw.csv for publisher (it is a blog name, and
# clues_raw.csv contains test-split rows we must not access — Decision 9).
id_map = pd.read_csv(
    SHARED_DATA / "id_map.csv",
    usecols=["clue_id", "puzzle_id"],
    keep_default_na=False, na_values=[""],
)
puzzle_metadata = pd.read_csv(
    SHARED_DATA / "puzzle_metadata.csv",
    usecols=["puzzle_id", "publisher"],
    keep_default_na=False, na_values=[""],
)

print(f"clues_wn_filtered:     {len(clues_wn_filtered):,} rows")
print(f"clues_val:             {len(clues_val):,} rows")
print(f"vocabulary_wndef:      {len(vocab_wndef):,} words")
print(f"f_clue.csv:            {len(f_clue_phrases):,} rows")
print(f"g_stock/f_clue_index:  {len(g_stock_f_clue_index):,} rows")
print(f"g1/f_clue_val_index:   {len(g1_f_clue_val_index):,} rows")
print(f"id_map:                {len(id_map):,} rows")
print(f"puzzle_metadata:       {len(puzzle_metadata):,} rows")
print(f"Load: {time.time() - t0:.1f}s")


In [ ]:
# === Load embedding arrays
t0 = time.time()

embeddings = {
    ("g1",      "f_common_wndef"): np.load(EMBED_DIR / "g1"      / "f_common_wndef.npy"),
    ("g_stock", "f_common_wndef"): np.load(EMBED_DIR / "g_stock" / "f_common_wndef.npy"),
    ("g1",      "f_clue_val"):     np.load(EMBED_DIR / "g1"      / "f_clue_val.npy"),
}
g_stock_f_clue_full = np.load(EMBED_DIR / "g_stock" / "f_clue.npy")

# Shape validation
assert embeddings[("g1", "f_common_wndef")].shape      == (len(vocab_wndef), 1024)
assert embeddings[("g_stock", "f_common_wndef")].shape == (len(vocab_wndef), 1024)
assert embeddings[("g1", "f_clue_val")].shape          == (len(g1_f_clue_val_index), 1024)
assert g_stock_f_clue_full.shape                        == (len(g_stock_f_clue_index), 1024)

print(f"g1/f_common_wndef:      {embeddings[('g1','f_common_wndef')].shape}")
print(f"g_stock/f_common_wndef: {embeddings[('g_stock','f_common_wndef')].shape}")
print(f"g1/f_clue_val:          {embeddings[('g1','f_clue_val')].shape}")
print(f"g_stock/f_clue (full):  {g_stock_f_clue_full.shape}")
print(f"Load + validate: {time.time() - t0:.1f}s")


In [ ]:
# === Align g_stock f_clue validation slice to g1's f_clue_val_index row order
# We build the g_stock val slice in the SAME row order as g1's index so that
# row i in both arrays corresponds to the same (clue_id, definition) pair.
t0 = time.time()

g_stock_clue_key_to_row = {
    (cid, defn): row
    for cid, defn, row in zip(
        g_stock_f_clue_index["clue_id"],
        g_stock_f_clue_index["definition"],
        g_stock_f_clue_index["row"],
    )
}
g_stock_val_rows = np.array([
    g_stock_clue_key_to_row[(cid, defn)]
    for cid, defn in zip(g1_f_clue_val_index["clue_id"],
                         g1_f_clue_val_index["definition"])
], dtype=np.int64)

embeddings[("g_stock", "f_clue_val")] = g_stock_f_clue_full[g_stock_val_rows]
assert embeddings[("g_stock", "f_clue_val")].shape == embeddings[("g1", "f_clue_val")].shape

clue_key_to_row = {
    (cid, defn): row
    for cid, defn, row in zip(
        g1_f_clue_val_index["clue_id"],
        g1_f_clue_val_index["definition"],
        g1_f_clue_val_index["row"],
    )
}
wndef_word_to_row = dict(zip(vocab_wndef["word"], vocab_wndef["row"]))

def rowwise_cosine(A, B):
    """Per-row cosine similarity between two equal-shape (N, D) arrays."""
    A_norm = A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-10)
    B_norm = B / (np.linalg.norm(B, axis=1, keepdims=True) + 1e-10)
    return np.sum(A_norm * B_norm, axis=1)

print(f"g_stock/f_clue_val (extracted): {embeddings[('g_stock','f_clue_val')].shape}")
print(f"Extract: {time.time() - t0:.1f}s")


In [ ]:
# === Full-dataset vs validation-slice norms for g_stock f_clue
t0 = time.time()

norms_full = np.linalg.norm(g_stock_f_clue_full, axis=1)
norms_val  = np.linalg.norm(embeddings[("g_stock", "f_clue_val")], axis=1)

def norm_summary(label, v):
    return {
        "Distribution": label,
        "N":      int(v.size),
        "Mean":   float(v.mean()),
        "Std":    float(v.std()),
        "P5":     float(np.percentile(v,  5)),
        "P25":    float(np.percentile(v, 25)),
        "P50":    float(np.percentile(v, 50)),
        "P75":    float(np.percentile(v, 75)),
        "P95":    float(np.percentile(v, 95)),
    }
norm_fullval_df = pd.DataFrame([
    norm_summary("g_stock f_clue (full, 239,406)", norms_full),
    norm_summary("g_stock f_clue (val,  47,933)",  norms_val),
])
with pd.option_context("display.float_format", "{:.4f}".format, "display.width", 140):
    print(norm_fullval_df.to_string(index=False))

print(f"Norm stats: {time.time() - t0:.1f}s")


In [ ]:
# === Figure: full-dataset vs validation-slice overlay (fclue_bimodal_norm_full_vs_val.png)
# Single axis. Both distributions on the same x-axis so bimodal peak
# locations are directly comparable. L2 -> diagonal hatching per FIGURE_STANDARDS.md.
fig, ax = plt.subplots(figsize=(8, 4.5))

bin_edges = np.linspace(
    min(norms_full.min(), norms_val.min()),
    max(norms_full.max(), norms_val.max()),
    61,
)
ax.hist(norms_full, bins=bin_edges, alpha=0.45, color="#4d4d4d",
        hatch="//", edgecolor="#4d4d4d",
        label="g_stock f_clue (full, 239,406)", density=True)
ax.hist(norms_val,  bins=bin_edges, alpha=0.55, color=MODEL_COLORS["g_stock"],
        hatch="//", edgecolor=MODEL_COLORS["g_stock"],
        label="g_stock f_clue (val,  47,933)", density=True)
ax.set_xlabel("L2 norm")
ax.set_ylabel("Density")
ax.set_title("g_stock f_clue norms: full dataset vs validation slice")
ax.grid(alpha=0.3)

ax.legend(loc="best", fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "fclue_bimodal_norm_full_vs_val.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# === Text histogram of g_stock f_clue norms for precise peak/valley identification
def text_hist(values, n_bins=25, lo=None, hi=None, width=50, label=""):
    """Print an ASCII horizontal histogram for a 1D array."""
    lo = float(values.min()) if lo is None else float(lo)
    hi = float(values.max()) if hi is None else float(hi)
    edges = np.linspace(lo, hi, n_bins + 1)
    counts, _ = np.histogram(values, bins=edges)
    m = counts.max() if counts.max() > 0 else 1
    print(f"{label}  (N={len(values):,}, range=[{lo:.2f}, {hi:.2f}])")
    for i, c in enumerate(counts):
        bar = "#" * int(round(width * c / m))
        print(f"  [{edges[i]:6.2f}, {edges[i+1]:6.2f})  {c:7,}  {bar}")

print("Full dataset (239,406):")
text_hist(norms_full, n_bins=25, lo=25.0, hi=34.0, label="g_stock f_clue (full)")
print()
print("Validation slice (47,933):")
text_hist(norms_val,  n_bins=25, lo=25.0, hi=34.0, label="g_stock f_clue (val)")


In [ ]:
# === Figure: g_stock vs g1 f_clue (val) norm overlay (fclue_bimodal_norm_overlay.png)
# Adapted from the right panel of g1be_norm_distributions.png. Same
# FIGURE_STANDARDS.md encoding: model color, "//" hatch for L2, density=True.
fig, ax = plt.subplots(figsize=(8, 4.5))

norms_val_by_model = {
    "g_stock": np.linalg.norm(embeddings[("g_stock", "f_clue_val")], axis=1),
    "g1":      np.linalg.norm(embeddings[("g1",      "f_clue_val")], axis=1),
}

lo = min(v.min() for v in norms_val_by_model.values())
hi = max(v.max() for v in norms_val_by_model.values())
bin_edges = np.linspace(lo, hi, 61)

for model in MODEL_NAMES:
    v = norms_val_by_model[model]
    ax.hist(v, bins=bin_edges, alpha=0.5, color=MODEL_COLORS[model],
            hatch="//", edgecolor=MODEL_COLORS[model],
            label=f"{model} (mean={v.mean():.2f}, std={v.std():.2f})",
            density=True)
    ax.axvline(v.mean(), color=MODEL_COLORS[model], linestyle="--", linewidth=1)

ax.set_xlabel("L2 norm")
ax.set_ylabel("Density")
ax.set_title("f_clue (val) L2 norms: g_stock (bimodal) vs g1 (unimodal)")
ax.grid(alpha=0.3)

ax.legend(loc="best", fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "fclue_bimodal_norm_overlay.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# === Figure: g_stock vs g1 f_clue (val) norm overlay — outline (fclue_bimodal_norm_overlay_outline.png)
# Presentation variant of fclue_bimodal_norm_overlay.png. Same data and bin
# edges; histtype="stepfilled" with low alpha gives a shaded region behind
# a step outline (no internal bar dividers, no mean lines).
fig, ax = plt.subplots(figsize=(8, 4.5))

lo = min(v.min() for v in norms_val_by_model.values())
hi = max(v.max() for v in norms_val_by_model.values())
bin_edges_overlay_outline = np.linspace(lo, hi, 61)

for model in MODEL_NAMES:
    v = norms_val_by_model[model]
    ax.hist(v, bins=bin_edges_overlay_outline, histtype="stepfilled",
            color=MODEL_COLORS[model], alpha=0.5, hatch="//",
            edgecolor=MODEL_COLORS[model], linewidth=1.5,
            label=model, density=True)

ax.set_xlabel("L2 norm")
ax.set_ylabel("Density")
ax.set_title("f_clue (val) L2 norms: g_stock (bimodal) vs g1 (unimodal)")
ax.grid(alpha=0.3)

ax.legend(loc="best", fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "fclue_bimodal_norm_overlay_outline.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# === Figure: g_stock and g1 f_clue (val) GMM fits (fclue_bimodal_norm_overlay_gmm.png)
# Presentation variant. Two panels show 1- and 2-component Gaussian fits
# side by side: the left panel is g_stock (bimodal — the 2-component fit
# captures structure the 1-component fit misses), the right panel is g1
# (unimodal — the two fits should largely coincide).
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

x_lo_gmm = min(v.min() for v in norms_val_by_model.values())
x_hi_gmm = max(v.max() for v in norms_val_by_model.values())
bin_edges_gmm_overlay = np.linspace(x_lo_gmm, x_hi_gmm, 61)

for ax, model in zip(axes, MODEL_NAMES):
    v = norms_val_by_model[model]
    ax.hist(v, bins=bin_edges_gmm_overlay, histtype="step",
            color=MODEL_COLORS[model], linewidth=1.5, density=True)

    x_curve = np.linspace(float(v.min()), float(v.max()), 300)

    # 1-component fit: dashed gray curve, dotted vertical at its mean.
    gm1 = GaussianMixture(n_components=1, random_state=RANDOM_STATE)
    gm1.fit(v.reshape(-1, 1))
    mu1c    = float(gm1.means_.ravel()[0])
    sigma1c = float(np.sqrt(gm1.covariances_.ravel()[0]))
    pdf1c   = sp_norm.pdf(x_curve, loc=mu1c, scale=sigma1c)
    ax.plot(x_curve, pdf1c, color="#888888", linewidth=1.5, linestyle="--")
    ax.axvline(mu1c, color="#888888", linewidth=1, linestyle=":")

    # 2-component fit: per-component dashed black, combined solid black.
    fit2 = fit_gmm2(v)
    for mu, sigma, w in [(fit2["mu1"], fit2["sigma1"], fit2["weight1"]),
                         (fit2["mu2"], fit2["sigma2"], fit2["weight2"])]:
        pdf = w * sp_norm.pdf(x_curve, loc=mu, scale=sigma)
        ax.plot(x_curve, pdf, color="black", linewidth=1.5, linestyle="--")
    pdf_total = (fit2["weight1"] * sp_norm.pdf(x_curve, loc=fit2["mu1"], scale=fit2["sigma1"])
               + fit2["weight2"] * sp_norm.pdf(x_curve, loc=fit2["mu2"], scale=fit2["sigma2"]))
    ax.plot(x_curve, pdf_total, color="black", linewidth=1.8, alpha=0.6)
    ax.axvline(fit2["mu1"], color="black", linewidth=1, linestyle=":")
    ax.axvline(fit2["mu2"], color="black", linewidth=1, linestyle=":")

    # Annotate all three means (1-component μ above, 2-component μ₁/μ₂ below).
    y_top = ax.get_ylim()[1]
    ax.annotate(f"μ = {mu1c:.2f}",
                xy=(mu1c, y_top * 0.92),
                fontsize=8, ha="center", color="#888888",
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.7))
    ax.annotate(f"μ₁ = {fit2['mu1']:.2f}",
                xy=(fit2["mu1"], y_top * 0.80),
                fontsize=8, ha="center", color="black",
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.7))
    ax.annotate(f"μ₂ = {fit2['mu2']:.2f}",
                xy=(fit2["mu2"], y_top * 0.80),
                fontsize=8, ha="center", color="black",
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.7))

    ax.set_xlim(x_lo_gmm, x_hi_gmm)
    ax.set_xlabel("L2 norm")
    ax.set_ylabel("Density")
    ax.set_title(f"{model} f_clue (val)")
    ax.grid(alpha=0.3)

fig.tight_layout()
fig.savefig(FIG_DIR / "fclue_bimodal_norm_overlay_gmm.png", dpi=300, bbox_inches="tight")
plt.show()


### §2 — Contrast: do uncontextualized vocabulary embeddings also show bimodality?

The f_clue bimodality could be specific to clue-contextualized embeddings,
where `<t></t>` tags interact with surrounding surface text, or it could be
a more general property of CALE embeddings. As a contrast, we check the
g_stock norm distributions for the two uncontextualized vocabulary f's:
`f_common_wndef` (53,930 words) and `f_common_wnex` (8,360 words). If
these are unimodal, the bimodality is specific to the clue-context
setting and our investigation in the remaining sections is scoped
correctly.

The wnex vocabulary is a strict subset of the wndef vocabulary (every
word with a WordNet usage example also has a definition), but the two
distributions are centered differently. To tell whether that centering
difference is driven by phrase construction (*f_common_wnex* vs.
*f_common_wndef*) or by the word subset itself, the right panel overlays
the f_wndef norms restricted to the 8,360 words that also appear in the
wnex vocabulary. If the restricted-wndef distribution matches the full
wndef distribution (centered ~29), the shift is driven by phrase
construction; if it matches the wnex distribution (centered ~30.6), it
is driven by which words are in the subset.

In [ ]:
# === g_stock vocabulary norms: f_wndef and f_wnex (fclue_bimodal_vocab_norms.png)
# Both panels overlay the f_wndef norms for the 8,360 wnex-subset words, so
# the reader can see (left) where wnex-eligible words sit within the full
# wndef norm distribution, and (right) whether the wnex/wndef centering
# difference is driven by phrase construction or by the word subset itself.
vocab_wnex = pd.read_csv(
    WN_DIR / "wnex" / "vocabulary_wnex.csv",
    keep_default_na=False, na_values=[""],
)
gstock_f_wnex = np.load(EMBED_DIR / "g_stock" / "f_common_wnex.npy")
assert gstock_f_wnex.shape == (len(vocab_wnex), 1024)

wndef_norms_contrast = np.linalg.norm(embeddings[("g_stock", "f_common_wndef")], axis=1)
wnex_norms_contrast  = np.linalg.norm(gstock_f_wnex, axis=1)

# Map each wnex word to its row in vocabulary_wndef, then pull the matching
# rows from the f_common_wndef embedding array.
word_to_wndef_row = {w: i for i, w in enumerate(vocab_wndef["word"])}
assert set(vocab_wnex["word"]).issubset(word_to_wndef_row),     "wnex vocabulary is expected to be a subset of wndef vocabulary"
wnex_in_wndef_rows = np.array([word_to_wndef_row[w] for w in vocab_wnex["word"]])
wndef_subset_emb   = embeddings[("g_stock", "f_common_wndef")][wnex_in_wndef_rows]
wndef_subset_norms = np.linalg.norm(wndef_subset_emb, axis=1)
assert wndef_subset_norms.shape == wnex_norms_contrast.shape

# Match x-axis range to g_stock f_clue (val) so peak positions are directly
# comparable across the three distributions.
x_lo, x_hi = float(norms_val.min()), float(norms_val.max())
bin_edges = np.linspace(x_lo, x_hi, 61)

SUBSET_COLOR = "#555555"  # dark gray — visually distinct from the filled blue
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Left panel: full f_wndef vocabulary, with the wnex-subset overlaid as an
# outline so the reader can see where wnex-eligible words sit within the
# full distribution.
ax = axes[0]
ax.hist(wndef_norms_contrast, bins=bin_edges, alpha=0.55,
        color=MODEL_COLORS["g_stock"], hatch="//",
        edgecolor=MODEL_COLORS["g_stock"], density=True,
        label=f"f_wndef (full, N={len(vocab_wndef):,})")
ax.hist(wndef_subset_norms, bins=bin_edges, histtype="step",
        color=SUBSET_COLOR, linewidth=1.5, density=True,
        label=f"f_wndef (wnex subset, N={len(vocab_wnex):,})")
ax.axvline(wndef_norms_contrast.mean(), color=MODEL_COLORS["g_stock"], linestyle="--", linewidth=1)
ax.axvline(wndef_subset_norms.mean(), color=SUBSET_COLOR, linestyle="--", linewidth=1)
ax.set_xlim(x_lo, x_hi)
ax.set_xlabel("L2 norm")
ax.set_ylabel("Density")
ax.set_title("g_stock f_wndef vocabulary norms")
ax.grid(alpha=0.3)

ax.legend(loc="best", fontsize=8)

# Right panel: f_wnex overlaid with f_wndef restricted to the same 8,360 words.
ax = axes[1]
ax.hist(wnex_norms_contrast, bins=bin_edges, alpha=0.5,
        color=MODEL_COLORS["g_stock"], hatch="//",
        edgecolor=MODEL_COLORS["g_stock"], density=True,
        label=f"f_wnex (N={len(vocab_wnex):,})")
ax.hist(wndef_subset_norms, bins=bin_edges, histtype="step",
        color=SUBSET_COLOR, linewidth=1.5, density=True,
        label="f_wndef (same words)")
ax.axvline(wnex_norms_contrast.mean(), color=MODEL_COLORS["g_stock"], linestyle="--", linewidth=1)
ax.axvline(wndef_subset_norms.mean(), color=SUBSET_COLOR, linestyle="--", linewidth=1)
ax.set_xlim(x_lo, x_hi)
ax.set_xlabel("L2 norm")
ax.set_ylabel("Density")
ax.set_title("g_stock f_wnex vs f_wndef (wnex subset)")
ax.grid(alpha=0.3)

ax.legend(loc="best", fontsize=8)

fig.tight_layout()
fig.savefig(FIG_DIR / "fclue_bimodal_vocab_norms.png", dpi=300, bbox_inches="tight")
plt.show()

vocab_norm_df = pd.DataFrame([
    {"Distribution": "g_stock f_wndef (53,930)",
     "Mean": float(wndef_norms_contrast.mean()),
     "Std":  float(wndef_norms_contrast.std()),
     "P5":   float(np.percentile(wndef_norms_contrast,  5)),
     "P50":  float(np.percentile(wndef_norms_contrast, 50)),
     "P95":  float(np.percentile(wndef_norms_contrast, 95))},
    {"Distribution": "g_stock f_wnex  (8,360)",
     "Mean": float(wnex_norms_contrast.mean()),
     "Std":  float(wnex_norms_contrast.std()),
     "P5":   float(np.percentile(wnex_norms_contrast,  5)),
     "P50":  float(np.percentile(wnex_norms_contrast, 50)),
     "P95":  float(np.percentile(wnex_norms_contrast, 95))},
    {"Distribution": "g_stock f_wndef restricted to wnex (8,360)",
     "Mean": float(wndef_subset_norms.mean()),
     "Std":  float(wndef_subset_norms.std()),
     "P5":   float(np.percentile(wndef_subset_norms,  5)),
     "P50":  float(np.percentile(wndef_subset_norms, 50)),
     "P95":  float(np.percentile(wndef_subset_norms, 95))},
])
with pd.option_context("display.float_format", "{:.4f}".format, "display.width", 140):
    print(vocab_norm_df.to_string(index=False))

In [ ]:
# === Figure: g_stock vocabulary norms — outline (fclue_bimodal_vocab_norms_outline.png)
# Presentation variant of fclue_bimodal_vocab_norms.png. Same data, bin
# edges, panel layout, and x-axis range, but every histogram is drawn as
# a step outline: the main distribution in g_stock blue, the wnex-subset
# overlay in SUBSET_COLOR.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Left panel: full f_wndef vocabulary, with the wnex-subset overlaid.
ax = axes[0]
ax.hist(wndef_norms_contrast, bins=bin_edges, histtype="step",
        color=MODEL_COLORS["g_stock"], linewidth=1.5, density=True,
        label=f"f_wndef (full, N={len(vocab_wndef):,}, mean={wndef_norms_contrast.mean():.2f})")
ax.hist(wndef_subset_norms, bins=bin_edges, histtype="step",
        color=SUBSET_COLOR, linewidth=1.5, density=True,
        label=f"f_wndef (wnex subset, N={len(vocab_wnex):,}, mean={wndef_subset_norms.mean():.2f})")
ax.axvline(wndef_norms_contrast.mean(), color=MODEL_COLORS["g_stock"], linestyle="--", linewidth=1)
ax.axvline(wndef_subset_norms.mean(), color=SUBSET_COLOR, linestyle="--", linewidth=1)
ax.set_xlim(x_lo, x_hi)
ax.set_xlabel("L2 norm")
ax.set_ylabel("Density")
ax.set_title("g_stock f_wndef vocabulary norms")
ax.grid(alpha=0.3)
ax.legend(loc="best", fontsize=8)

# Right panel: f_wnex overlaid with f_wndef restricted to the same words.
ax = axes[1]
ax.hist(wnex_norms_contrast, bins=bin_edges, histtype="step",
        color=MODEL_COLORS["g_stock"], linewidth=1.5, density=True,
        label=f"f_wnex (N={len(vocab_wnex):,}, mean={wnex_norms_contrast.mean():.2f})")
ax.hist(wndef_subset_norms, bins=bin_edges, histtype="step",
        color=SUBSET_COLOR, linewidth=1.5, density=True,
        label=f"f_wndef (same words, mean={wndef_subset_norms.mean():.2f})")
ax.axvline(wnex_norms_contrast.mean(), color=MODEL_COLORS["g_stock"], linestyle="--", linewidth=1)
ax.axvline(wndef_subset_norms.mean(), color=SUBSET_COLOR, linestyle="--", linewidth=1)
ax.set_xlim(x_lo, x_hi)
ax.set_xlabel("L2 norm")
ax.set_ylabel("Density")
ax.set_title("g_stock f_wnex vs f_wndef (wnex subset)")
ax.grid(alpha=0.3)
ax.legend(loc="best", fontsize=8)

fig.tight_layout()
fig.savefig(FIG_DIR / "fclue_bimodal_vocab_norms_outline.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# === Figure: g1 vocabulary norms (fclue_bimodal_vocab_norms_g1.png)
# Presentation variant: same 2-panel layout and same bin edges / x-axis
# range as the g_stock vocab norms figure, but using g1 embeddings
# everywhere. Loads g1/f_common_wnex.npy fresh; g1 wndef norms are
# derived from the embeddings already in memory.
g1_f_wnex = np.load(EMBED_DIR / "g1" / "f_common_wnex.npy")
assert g1_f_wnex.shape == (len(vocab_wnex), 1024)

g1_wndef_norms_contrast = np.linalg.norm(embeddings[("g1", "f_common_wndef")], axis=1)
g1_wnex_norms_contrast  = np.linalg.norm(g1_f_wnex, axis=1)
g1_wndef_subset_norms   = np.linalg.norm(
    embeddings[("g1", "f_common_wndef")][wnex_in_wndef_rows], axis=1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Left panel: full f_wndef vocabulary, with the wnex-subset overlaid.
ax = axes[0]
ax.hist(g1_wndef_norms_contrast, bins=bin_edges, histtype="step",
        color=MODEL_COLORS["g1"], linewidth=1.5, density=True,
        label=f"f_wndef (full, N={len(vocab_wndef):,}, mean={g1_wndef_norms_contrast.mean():.2f})")
ax.hist(g1_wndef_subset_norms, bins=bin_edges, histtype="step",
        color=SUBSET_COLOR, linewidth=1.5, density=True,
        label=f"f_wndef (wnex subset, N={len(vocab_wnex):,}, mean={g1_wndef_subset_norms.mean():.2f})")
ax.axvline(g1_wndef_norms_contrast.mean(), color=MODEL_COLORS["g1"], linestyle="--", linewidth=1)
ax.axvline(g1_wndef_subset_norms.mean(), color=SUBSET_COLOR, linestyle="--", linewidth=1)
ax.set_xlim(x_lo, x_hi)
ax.set_xlabel("L2 norm")
ax.set_ylabel("Density")
ax.set_title("g1 f_wndef vocabulary norms")
ax.grid(alpha=0.3)
ax.legend(loc="best", fontsize=8)

# Right panel: f_wnex overlaid with f_wndef restricted to the same words.
ax = axes[1]
ax.hist(g1_wnex_norms_contrast, bins=bin_edges, histtype="step",
        color=MODEL_COLORS["g1"], linewidth=1.5, density=True,
        label=f"f_wnex (N={len(vocab_wnex):,}, mean={g1_wnex_norms_contrast.mean():.2f})")
ax.hist(g1_wndef_subset_norms, bins=bin_edges, histtype="step",
        color=SUBSET_COLOR, linewidth=1.5, density=True,
        label=f"f_wndef (same words, mean={g1_wndef_subset_norms.mean():.2f})")
ax.axvline(g1_wnex_norms_contrast.mean(), color=MODEL_COLORS["g1"], linestyle="--", linewidth=1)
ax.axvline(g1_wndef_subset_norms.mean(), color=SUBSET_COLOR, linestyle="--", linewidth=1)
ax.set_xlim(x_lo, x_hi)
ax.set_xlabel("L2 norm")
ax.set_ylabel("Density")
ax.set_title("g1 f_wnex vs f_wndef (wnex subset)")
ax.grid(alpha=0.3)
ax.legend(loc="best", fontsize=8)

fig.tight_layout()
fig.savefig(FIG_DIR / "fclue_bimodal_vocab_norms_g1.png", dpi=300, bbox_inches="tight")
plt.show()


### §2 — Cross-format directional alignment: do wndef and wnex embeddings *point* the same way?

The previous panel showed that the wndef-vs-wnex norm gap (~1.6 L2 units)
is driven by phrase format rather than by the words themselves. But norm
captures only magnitude. The angular question is just as important: for
the same word, does CALE place its dictionary-definition phrase and its
usage-example phrase in the same direction in embedding space, or in
different directions?

If mean cosine is near 1.0, the two formats produce nearly parallel
vectors that differ mostly in length, and a model trained on one format
should generalize cleanly to the other. A noticeably lower mean cosine
would mean the two formats push the same word into genuinely different
regions of the space, with implications for whether wndef-trained
behavior carries over to wnex.

We compute this for both g_stock and g1 to see whether fine-tuning on
clue-contextualized triplets has changed the directional relationship
between the two uncontextualized phrase formats.

In [ ]:
# === Cross-format directional alignment: f_wndef vs f_wnex for the same 8,360 words
# For each word in the wnex vocabulary, compare its f_wndef and f_wnex
# embeddings under each model. wndef_subset_emb (from cell 14) is already
# row-aligned to vocab_wnex order via wnex_in_wndef_rows; we apply the
# same row mapping to g1's f_common_wndef so g1's two arrays line up too.
gstock_cross_cos = rowwise_cosine(wndef_subset_emb, gstock_f_wnex)

g1_f_wnex = np.load(EMBED_DIR / "g1" / "f_common_wnex.npy")
assert g1_f_wnex.shape == (len(vocab_wnex), 1024)
g1_wndef_subset_emb = embeddings[("g1", "f_common_wndef")][wnex_in_wndef_rows]
g1_cross_cos = rowwise_cosine(g1_wndef_subset_emb, g1_f_wnex)

cross_cos_df = pd.DataFrame([
    {"Model": "g_stock", "N": int(len(gstock_cross_cos)),
     "Mean":   float(gstock_cross_cos.mean()),
     "Median": float(np.median(gstock_cross_cos)),
     "Std":    float(gstock_cross_cos.std()),
     "P5":     float(np.percentile(gstock_cross_cos,  5)),
     "P95":    float(np.percentile(gstock_cross_cos, 95))},
    {"Model": "g1", "N": int(len(g1_cross_cos)),
     "Mean":   float(g1_cross_cos.mean()),
     "Median": float(np.median(g1_cross_cos)),
     "Std":    float(g1_cross_cos.std()),
     "P5":     float(np.percentile(g1_cross_cos,  5)),
     "P95":    float(np.percentile(g1_cross_cos, 95))},
])
with pd.option_context("display.float_format", "{:.4f}".format, "display.width", 140):
    print(cross_cos_df.to_string(index=False))

In [ ]:
# === Figure: cross-format directional alignment (fclue_bimodal_cross_format_cosine.png)
# Shared bin edges spanning the union of both distributions, so the two
# histograms are directly comparable. The lower edge is the global min
# across both models; the upper edge is 1.0 (perfect alignment).
x_lo = float(min(gstock_cross_cos.min(), g1_cross_cos.min()))
bin_edges = np.linspace(x_lo, 1.0, 61)

fig, ax = plt.subplots(figsize=(7, 4.5))

# density=True so the GMM PDF curves are on the same vertical scale.
ax.hist(gstock_cross_cos, bins=bin_edges, alpha=0.55,
        color=MODEL_COLORS["g_stock"], density=True,
        label=f"g_stock (mean={gstock_cross_cos.mean():.3f})")
ax.hist(g1_cross_cos, bins=bin_edges, alpha=0.55,
        color=MODEL_COLORS["g1"], density=True,
        label=f"g1 (mean={g1_cross_cos.mean():.3f})")
ax.axvline(gstock_cross_cos.mean(), color=MODEL_COLORS["g_stock"],
           linestyle="--", linewidth=1)
ax.axvline(g1_cross_cos.mean(), color=MODEL_COLORS["g1"],
           linestyle="--", linewidth=1)

ax.set_xlabel("cosine similarity (same word, wndef vs wnex)")
ax.set_ylabel("density")
ax.set_title(f"Cross-format directional alignment ({len(vocab_wnex):,} words)")
ax.grid(alpha=0.3)

ax.legend(loc="best", fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "fclue_bimodal_cross_format_cosine.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# === Assemble eval pairs: (clue_row, def_row, ans_row) for validation
# Same key-matching approach as g1_basic_evaluation. Rows missing clue,
# def, or answer embeddings are dropped.
t0 = time.time()

eval_clue_rows = np.empty(len(clues_val), dtype=np.int64)
eval_def_rows  = np.empty(len(clues_val), dtype=np.int64)
eval_ans_rows  = np.empty(len(clues_val), dtype=np.int64)
n_miss = {"clue": 0, "def": 0, "ans": 0}

for i, (cid, orig_def, def_wn, ans_wn) in enumerate(zip(
    clues_val["clue_id"], clues_val["definition"],
    clues_val["definition_wn"], clues_val["answer_wn"],
)):
    eval_clue_rows[i] = clue_key_to_row.get((cid, orig_def), -1)
    eval_def_rows[i]  = wndef_word_to_row.get(def_wn,  -1)
    eval_ans_rows[i]  = wndef_word_to_row.get(ans_wn,  -1)
    if eval_clue_rows[i] < 0: n_miss["clue"] += 1
    if eval_def_rows[i]  < 0: n_miss["def"]  += 1
    if eval_ans_rows[i]  < 0: n_miss["ans"]  += 1

keep_mask = (eval_clue_rows >= 0) & (eval_def_rows >= 0) & (eval_ans_rows >= 0)
eval_clue_rows = eval_clue_rows[keep_mask]
eval_def_rows  = eval_def_rows[keep_mask]
eval_ans_rows  = eval_ans_rows[keep_mask]
clues_val_kept = clues_val[keep_mask].reset_index(drop=True).copy()

print(f"clues_val rows:       {len(clues_val):,}")
print(f"  missing clue row:   {n_miss['clue']:,}")
print(f"  missing def row:    {n_miss['def']:,}")
print(f"  missing ans row:    {n_miss['ans']:,}")
print(f"  kept:               {len(eval_clue_rows):,} "
      f"({len(eval_clue_rows)/len(clues_val):.1%})")
print(f"Assemble: {time.time() - t0:.1f}s")


In [ ]:
# === T=0 / T=1 cosine and L2 for g_stock on the evaluation pairs
t0 = time.time()

def t0_t1(model):
    clue_emb  = embeddings[(model, "f_clue_val")]
    vocab_emb = embeddings[(model, "f_common_wndef")]
    T0_cos = rowwise_cosine(vocab_emb[eval_def_rows], vocab_emb[eval_ans_rows])
    T1_cos = rowwise_cosine(clue_emb[eval_clue_rows], vocab_emb[eval_ans_rows])
    T0_l2  = np.linalg.norm(vocab_emb[eval_def_rows] - vocab_emb[eval_ans_rows], axis=1)
    T1_l2  = np.linalg.norm(clue_emb[eval_clue_rows] - vocab_emb[eval_ans_rows], axis=1)
    return {"T0_cos": T0_cos, "T1_cos": T1_cos, "T0_l2": T0_l2, "T1_l2": T1_l2}

t01 = {m: t0_t1(m) for m in MODEL_NAMES}
t01_rows = []
for m in MODEL_NAMES:
    for k, v in t01[m].items():
        t01_rows.append({
            "Model": m, "Metric": k,
            "Mean":   float(v.mean()),
            "Median": float(np.median(v)),
            "Std":    float(v.std()),
            "P5":     float(np.percentile(v,  5)),
            "P95":    float(np.percentile(v, 95)),
        })
t01_df = pd.DataFrame(t01_rows)
with pd.option_context("display.float_format", "{:.4f}".format, "display.width", 140):
    print(t01_df.to_string(index=False))
print(f"T=0/T=1: {time.time() - t0:.1f}s")


In [ ]:
# === Figure: g_stock T=0/T=1 cosine and L2 (fclue_bimodal_t0_t1.png)
# T=0 outlined, T=1 filled, both in g_stock blue. L2 panel additionally hatched.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
c = MODEL_COLORS["g_stock"]

# Left: cosine
T0c, T1c = t01["g_stock"]["T0_cos"], t01["g_stock"]["T1_cos"]
lo, hi = float(np.percentile(np.concatenate([T0c, T1c]), 0.5)), float(np.percentile(np.concatenate([T0c, T1c]), 99.5))
cos_bins = np.linspace(lo, hi, 61)
axes[0].hist(T1c, bins=cos_bins, histtype="stepfilled", color=c,
             alpha=0.55, label="T=1 (filled)", density=True)
axes[0].hist(T0c, bins=cos_bins, histtype="step",       color=c,
             linewidth=1.8, label="T=0 (outline)", density=True)
axes[0].axvline(T0c.mean(), color=c, linestyle="--", linewidth=1)
axes[0].axvline(T1c.mean(), color=c, linestyle="-",  linewidth=1)
axes[0].set_xlim(lo, hi)
axes[0].set_xlabel("Cosine similarity")
axes[0].set_ylabel("Density")
axes[0].set_title("g_stock: T=0 vs T=1 (cosine)")
axes[0].grid(alpha=0.3)
axes[0].legend(loc="best", fontsize=8)

# Right: L2
T0l, T1l = t01["g_stock"]["T0_l2"], t01["g_stock"]["T1_l2"]
lo, hi = float(np.percentile(np.concatenate([T0l, T1l]), 0.5)), float(np.percentile(np.concatenate([T0l, T1l]), 99.5))
l2_bins = np.linspace(lo, hi, 61)
axes[1].hist(T1l, bins=l2_bins, histtype="stepfilled", color=c,
             alpha=0.55, hatch="//", edgecolor=c,
             label="T=1 (filled)", density=True)
axes[1].hist(T0l, bins=l2_bins, histtype="step",       color=c,
             linewidth=1.8, label="T=0 (outline)", density=True)
axes[1].axvline(T0l.mean(), color=c, linestyle="--", linewidth=1)
axes[1].axvline(T1l.mean(), color=c, linestyle="-",  linewidth=1)
axes[1].set_xlim(lo, hi)
axes[1].set_xlabel("L2 distance")
axes[1].set_ylabel("Density")
axes[1].set_title("g_stock: T=0 vs T=1 (L2)")
axes[1].grid(alpha=0.3)
axes[1].legend(loc="best", fontsize=8)

fig.tight_layout()
fig.savefig(FIG_DIR / "fclue_bimodal_t0_t1.png", dpi=300, bbox_inches="tight")
plt.show()

## §3 — Definition position modulates which norm regime dominates

The definition in a cryptic clue normally appears at either the start or
end of the surface, occasionally in the middle. When at the start, the
tagged f_clue phrase looks like `<t>Definition</t> rest of clue`; when at
the end, `rest of clue <t>definition</t>`. Transformer attention patterns
are position-sensitive, so the same word tagged in different positions may
produce embeddings with different properties.

The finding: CALE has **two norm regimes for tagged text**, at roughly
29.5 and 31.5, and **both regimes are present regardless of definition
position**. What position changes is the balance of which regime
dominates: start-definitions preferentially land in the upper-norm mode,
end-definitions preferentially land in the lower-norm mode. The mode
locations themselves do not move.

The overall bimodality in §2 is visible because start-definitions
(N=26,964) outnumber end-definitions (N=20,459), giving extra weight to
the upper mode. We classify each validation clue's definition position
and compare norm distributions across position groups to make this
visible.


In [ ]:
# === Classify every validation clue by definition position within surface
t0 = time.time()

def classify_position(surface, definition):
    s = surface.lower().strip()
    d = definition.lower().strip()
    if not s or not d:
        return "unknown"
    if s.startswith(d):
        return "start"
    if s.endswith(d):
        return "end"
    return "middle"

pos = np.array([
    classify_position(s, d)
    for s, d in zip(clues_val_kept["surface"], clues_val_kept["definition"])
])
clues_val_kept["def_position"] = pos

# f_clue norms aligned to clues_val_kept (g_stock val slice, after keep_mask).
# Row i of clues_val_kept corresponds to eval_clue_rows[i] in the f_clue_val
# array, so we re-index.
fclue_norms_gstock = np.linalg.norm(
    embeddings[("g_stock", "f_clue_val")][eval_clue_rows], axis=1
)
fclue_norms_g1 = np.linalg.norm(
    embeddings[("g1",      "f_clue_val")][eval_clue_rows], axis=1
)
clues_val_kept["fclue_norm_gstock"] = fclue_norms_gstock
clues_val_kept["fclue_norm_g1"]     = fclue_norms_g1

pos_counts = clues_val_kept["def_position"].value_counts()
print("Definition position counts:")
for k, n in pos_counts.items():
    print(f"  {k:6s}: {n:6,} ({n/len(clues_val_kept):.1%})")
print(f"Classify: {time.time() - t0:.1f}s")


In [ ]:
# === Figure: f_clue norms by definition position (fclue_bimodal_by_position.png)
# Two panels: start vs end. g_stock blue with L2 hatching. A 2-component
# GMM is fit per panel and overlaid in dark blue so the two panels' peak
# locations can be compared at a glance.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

lo, hi = fclue_norms_gstock.min(), fclue_norms_gstock.max()
bin_edges = np.linspace(lo, hi, 61)

position_fits = {}
for ax, which in zip(axes, ["start", "end"]):
    v = clues_val_kept.loc[clues_val_kept["def_position"] == which, "fclue_norm_gstock"].to_numpy()
    ax.hist(v, bins=bin_edges, alpha=0.55, color=MODEL_COLORS["g_stock"],
            hatch="//", edgecolor=MODEL_COLORS["g_stock"],
            density=True, label=f"{which} (N={len(v):,})")
    ax.axvline(v.mean(), color=MODEL_COLORS["g_stock"], linestyle="--", linewidth=1)
    ax.set_xlabel("L2 norm")
    ax.set_ylabel("Density")
    ax.set_title(f"g_stock f_clue norms — {which}-of-surface definitions")
    ax.grid(alpha=0.3)

    fit = fit_gmm2(v)
    position_fits[which] = fit
    overlay_gmm2(ax, v, fit, color="#0b3d91", annotate_y_frac=0.92)
    ax.legend(loc="best", fontsize=8)

fig.tight_layout()
fig.savefig(FIG_DIR / "fclue_bimodal_by_position.png", dpi=300, bbox_inches="tight")
plt.show()

for which, fit in position_fits.items():
    gmm_fits.append({"figure": "fclue_bimodal_by_position",
                     "distribution": f"g_stock f_clue norms ({which}-defs)",
                     **fit})


In [ ]:
# === Text histograms + summary stats by definition position
pos_stats = []
for which in ["start", "end", "middle"]:
    v = clues_val_kept.loc[clues_val_kept["def_position"] == which, "fclue_norm_gstock"].to_numpy()
    if len(v) == 0:
        continue
    pos_stats.append({
        "Position": which,
        "N":    int(len(v)),
        "Mean": float(v.mean()),
        "Std":  float(v.std()),
        "P25":  float(np.percentile(v, 25)),
        "P50":  float(np.percentile(v, 50)),
        "P75":  float(np.percentile(v, 75)),
    })
pos_stats_df = pd.DataFrame(pos_stats)
with pd.option_context("display.float_format", "{:.4f}".format, "display.width", 140):
    print(pos_stats_df.to_string(index=False))
print()
for which in ["start", "end"]:
    v = clues_val_kept.loc[clues_val_kept["def_position"] == which, "fclue_norm_gstock"].to_numpy()
    text_hist(v, n_bins=25, lo=25.0, hi=34.0, label=f"g_stock f_clue norms — {which}")
    print()


## §4 — Ruling out alternative explanations

For start-definitions specifically (since they carry the bimodality), we
split at the valley (norm ≈ 30.5) into "lower mode" and "upper mode"
groups and test whether any observable text property explains which mode a
clue lands in. The question in every comparison below is the same:
**does this variable cleanly separate the two modes?** If the answer is no
across the board, the assignment is driven by something internal to the
model, not by text properties.


In [ ]:
# === Split start-definitions at the valley norm = 30.5
VALLEY = 30.5
start_mask = clues_val_kept["def_position"] == "start"
starts = clues_val_kept[start_mask].copy().reset_index(drop=True)
starts["norm_group"] = np.where(starts["fclue_norm_gstock"] < VALLEY, "lower", "upper")

# Row index into the f_clue_val arrays for start-defs only (for §5 and §6).
start_fclue_rows = eval_clue_rows[start_mask.to_numpy()]
start_def_rows   = eval_def_rows[start_mask.to_numpy()]
start_ans_rows   = eval_ans_rows[start_mask.to_numpy()]

ng = starts["norm_group"].value_counts()
print(f"start-defs total:       {len(starts):,}")
print(f"  lower (norm < 30.5):  {ng.get('lower', 0):,}")
print(f"  upper (norm >= 30.5): {ng.get('upper', 0):,}")


In [ ]:
# === Text property comparisons: word count, definition length
def group_means(df, col):
    return df.groupby("norm_group")[col].agg(["mean", "std", "count"])

starts["surface_word_count"] = starts["surface"].str.split().str.len()
starts["definition_char_len"] = starts["definition"].str.len()

print("Surface word count by norm group:")
print(group_means(starts, "surface_word_count").round(3))
print()
print("Definition character length by norm group:")
print(group_means(starts, "definition_char_len").round(3))


In [ ]:
# === Definition subword token count using MBERT tokenizer
# Network dependency (one-time download, cached).
from transformers import AutoTokenizer

t0 = time.time()
tok = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")

unique_defs = starts["definition"].unique()
def_token_counts = {
    d: len(tok.encode(d, add_special_tokens=False)) for d in unique_defs
}
starts["def_subword_tokens"] = starts["definition"].map(def_token_counts)
print(f"Tokenize {len(unique_defs):,} unique definitions: {time.time() - t0:.1f}s")
print()

print("Subword token count by norm group:")
print(group_means(starts, "def_subword_tokens").round(3))
print()
print("Token count distribution (start-defs):")
print(starts["def_subword_tokens"].value_counts().sort_index())


In [ ]:
# === Figure: norm distribution for 1-token vs 2-token definitions (fclue_bimodal_by_subword_tokens.png)
# Two panels by MBERT-tokenizer subword count. A 2-component GMM is fit per
# panel; if a panel is empty (N=0), no fit is performed.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

lo, hi = starts["fclue_norm_gstock"].min(), starts["fclue_norm_gstock"].max()
bin_edges = np.linspace(lo, hi, 61)

token_fits = {}
for ax, tk in zip(axes, [1, 2]):
    v = starts.loc[starts["def_subword_tokens"] == tk, "fclue_norm_gstock"].to_numpy()
    if len(v) == 0:
        ax.set_title(f"start-defs, {tk}-token (N=0)")
        ax.set_axis_off()
        continue
    ax.hist(v, bins=bin_edges, alpha=0.55, color=MODEL_COLORS["g_stock"],
            hatch="//", edgecolor=MODEL_COLORS["g_stock"],
            density=True, label=f"{tk}-token defs (N={len(v):,})")
    ax.axvline(v.mean(), color=MODEL_COLORS["g_stock"], linestyle="--", linewidth=1)
    ax.axvline(VALLEY, color="black", linestyle=":", linewidth=1, label=f"valley = {VALLEY}")
    ax.set_xlabel("L2 norm")
    ax.set_ylabel("Density")
    ax.set_title(f"start-defs, {tk}-subword-token definitions")
    ax.grid(alpha=0.3)

    fit = fit_gmm2(v)
    token_fits[tk] = fit
    overlay_gmm2(ax, v, fit, color="#0b3d91", annotate_y_frac=0.92)
    ax.legend(loc="best", fontsize=8)

fig.tight_layout()
fig.savefig(FIG_DIR / "fclue_bimodal_by_subword_tokens.png", dpi=300, bbox_inches="tight")
plt.show()

for tk, fit in token_fits.items():
    gmm_fits.append({"figure": "fclue_bimodal_by_subword_tokens",
                     "distribution": f"g_stock f_clue norms ({tk}-token start-defs)",
                     **fit})


In [ ]:
# === Publisher comparison (via id_map -> puzzle_metadata, DATA_RAW.md §4.3)
# starts carries clue_id; join clue_id -> puzzle_id -> publisher. Never
# use clues_raw.csv for publisher — that file contains test-split data
# (Decision 9) and its `source` column is a blog name, not a publisher.
starts_pub = (
    starts
    .merge(id_map,          on="clue_id",  how="left")
    .merge(puzzle_metadata, on="puzzle_id", how="left")
)
n_missing_pub = starts_pub["publisher"].isna().sum()
print(f"start-defs with missing publisher: {n_missing_pub}/{len(starts_pub)}")

pub_stats = (
    starts_pub.groupby("publisher")
    .agg(
        N=("fclue_norm_gstock", "size"),
        mean=("fclue_norm_gstock", "mean"),
        std=("fclue_norm_gstock", "std"),
    )
    .sort_values("N", ascending=False)
)
with pd.option_context("display.float_format", "{:.4f}".format, "display.width", 140):
    print(pub_stats.head(10).to_string())


In [ ]:
# === Figure: norm distributions for the top 3 publishers (fclue_bimodal_by_source.png)
# A 2-component GMM is fit per panel so peak locations can be compared
# across publishers — if all three publishers exhibit the same μ values,
# that confirms the bimodality is publisher-agnostic.
top_publishers = pub_stats.head(3).index.tolist()
print(f"Top 3 publishers: {top_publishers}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
lo, hi = starts["fclue_norm_gstock"].min(), starts["fclue_norm_gstock"].max()
bin_edges = np.linspace(lo, hi, 61)

publisher_fits = {}
for ax, pub in zip(axes, top_publishers):
    v = starts_pub.loc[starts_pub["publisher"] == pub, "fclue_norm_gstock"].to_numpy()
    ax.hist(v, bins=bin_edges, alpha=0.55, color=MODEL_COLORS["g_stock"],
            hatch="//", edgecolor=MODEL_COLORS["g_stock"],
            density=True, label=f"N={len(v):,}")
    ax.axvline(v.mean(), color=MODEL_COLORS["g_stock"], linestyle="--", linewidth=1)
    ax.axvline(VALLEY, color="black", linestyle=":", linewidth=1, label=f"valley = {VALLEY}")
    ax.set_xlabel("L2 norm")
    ax.set_ylabel("Density")
    ax.set_title(f"publisher = {pub}")
    ax.grid(alpha=0.3)

    fit = fit_gmm2(v)
    publisher_fits[pub] = fit
    overlay_gmm2(ax, v, fit, color="#0b3d91", annotate_y_frac=0.92)
    ax.legend(loc="best", fontsize=8)

fig.tight_layout()
fig.savefig(FIG_DIR / "fclue_bimodal_by_source.png", dpi=300, bbox_inches="tight")
plt.show()

for pub, fit in publisher_fits.items():
    gmm_fits.append({"figure": "fclue_bimodal_by_source",
                     "distribution": f"g_stock f_clue norms (start-defs, publisher={pub})",
                     **fit})


### §4 — Wordplay type stratification

`wordplay_metadata.csv` provides algorithmically verified boolean flags for
each clue identifying whether it exhibits structural patterns characteristic
of common cryptic wordplay devices: anagrams (single-word and
consecutive-word), hidden words (forward and reverse), various letter-
selection patterns, and double definitions. These flags reflect *pattern
detection*, not setter intent — `True` means the surface contains the
structural cue, not that the setter actually intended that device.

We use these flags to ask: does any single wordplay device account for the
upper-mode mass we see in §3 and §4? If, say, double-definition clues all
landed in the upper mode, that would be a clean text-side explanation for
the bimodality. Several wordplay types occur in fewer than 1% of clues, so
the validation slice alone (47,933 rows) is too small for stable subset
GMMs. We therefore run this stratification on the full 239,406-row f_clue
embedding array. For each subset, we fit a 2-component GMM (or use posterior
weights from the full-dataset reference fit when N is small) and compare
mixture weights between True-clues and False-clues at fixed mode locations.


In [ ]:
# === Wordplay metadata join + position classification (full dataset)
# Boolean flags per clue_id from the structural-pattern detector. We left-
# join onto f_clue_index (which is keyed by row position in the full
# 239,406-row embedding array) and verify nothing falls out.
t0 = time.time()

wordplay = pd.read_csv(
    SHARED_DATA / "wordplay_metadata.csv",
    keep_default_na=False, na_values=[""],
)
wordplay_cols = [c for c in wordplay.columns if c != "clue_id"]

# f_clue_index row order = row order of the full f_clue.npy embedding array;
# norms_full (computed in §2) is aligned to it 1:1.
fclue_full_df = g_stock_f_clue_index.merge(wordplay, on="clue_id", how="left")
assert len(fclue_full_df) == len(g_stock_f_clue_index), \
    "Left join changed row count — wordplay_metadata may have duplicate clue_ids."
n_unmatched = fclue_full_df[wordplay_cols[0]].isna().sum()
assert n_unmatched == 0, f"{n_unmatched} f_clue rows lack wordplay metadata."

# Attach the precomputed full-dataset norm vector (same row order).
assert len(norms_full) == len(fclue_full_df)
fclue_full_df["fclue_norm_gstock"] = norms_full

# Definition position requires the surface text. clues_wn_filtered is keyed
# by (clue_id, definition) after multi-def expansion, so we join on that
# composite key — clue_id alone is not unique.
fclue_full_df = fclue_full_df.merge(
    clues_wn_filtered[["clue_id", "definition", "surface"]],
    on=["clue_id", "definition"], how="left",
)
assert fclue_full_df["surface"].notna().all(), \
    "Some (clue_id, definition) pairs missing from clues_wn_filtered."

fclue_full_df["def_position"] = [
    classify_position(s, d)
    for s, d in zip(fclue_full_df["surface"], fclue_full_df["definition"])
]

# Wordplay frequencies; drop columns with too few True observations to fit
# a stable 2-component GMM.
true_counts = fclue_full_df[wordplay_cols].sum().astype(int).sort_values(ascending=False)
print("Wordplay True-counts across the full f_clue dataset:")
for col, n in true_counts.items():
    print(f"  {col:24s}: {n:6,} ({n/len(fclue_full_df):.2%})")

MIN_TRUE = 50
wordplay_cols_kept = [c for c in wordplay_cols if true_counts[c] >= MIN_TRUE]
dropped = [c for c in wordplay_cols if c not in wordplay_cols_kept]
if dropped:
    print(f"\nDropping {len(dropped)} wordplay types with N(True) < {MIN_TRUE}: {dropped}")
print(f"\nWordplay types analyzed: {len(wordplay_cols_kept)}")
print(f"Join + classify: {time.time() - t0:.1f}s")


In [ ]:
# === Reference 2-component GMM on the full 239,406 f_clue norms
# This fixes the canonical mode locations (mu1, mu2). Subset GMMs below are
# compared against these values so we can confirm that wordplay-stratified
# distributions shift the mixture weight without moving the modes.
full_gmm = GaussianMixture(n_components=2, random_state=RANDOM_STATE).fit(
    norms_full.reshape(-1, 1)
)
# Sort components by mean so (mu1, sigma1, weight1) is always the lower mode.
order        = np.argsort(full_gmm.means_.ravel())
full_mus     = full_gmm.means_.ravel()[order]
full_sigmas  = np.sqrt(full_gmm.covariances_.ravel())[order]
full_weights = full_gmm.weights_[order]
full_mu1, full_mu2 = float(full_mus[0]), float(full_mus[1])
full_w1,  full_w2  = float(full_weights[0]), float(full_weights[1])

print(f"Full-dataset reference GMM:")
print(f"  mu1={full_mu1:.4f}  sigma1={full_sigmas[0]:.4f}  pi1={full_w1:.4f}")
print(f"  mu2={full_mu2:.4f}  sigma2={full_sigmas[1]:.4f}  pi2={full_w2:.4f}")


In [ ]:
# === Per-wordplay-type 2-component GMM (free fit) or posterior reweighting
# Two-tier approach: for subsets with at least N=200 True observations, fit
# a fresh GMM (free fit). For smaller subsets, use the full-dataset GMM's
# posteriors averaged across observations (constrained: means inherited).
# This produces a comparable pi1 estimate when subset size is too small
# for a stable independent fit.
FREE_FIT_MIN_N = 200

# Letterplay = the wordplay devices that operate at the letter level
# (anagrams, hidden words, selection patterns), excluding double_def which
# is a definition-level rather than letter-level construction. Used to
# build the "any letterplay" and "no letterplay" synthetic subsets, which
# together with double_def partition the full clue population.
LETTERPLAY_COLS = [
    "anagram_single_word", "anagram_consec_words",
    "hidden_fwd", "hidden_rev",
    "selection_alt", "selection_alt_rev",
    "selection_firsts", "selection_firsts_rev",
    "selection_lasts", "selection_lasts_rev",
]

def _free_fit_pi1(values):
    """Free 2-component GMM fit; returns (mu1, mu2, pi1, full_fit)."""
    fit = fit_gmm2(values)
    return fit["mu1"], fit["mu2"], fit["weight1"], fit

def _constrained_pi1(values):
    """Average posterior probability of component 1 (lower mode) under the
    full-dataset GMM. This is the maximum-likelihood mixture weight when
    component parameters are held fixed at the reference fit."""
    proba = full_gmm.predict_proba(values.reshape(-1, 1))
    # Map full_gmm component index to sorted-by-mean order.
    order_full = np.argsort(full_gmm.means_.ravel())
    pi1 = float(proba[:, order_full[0]].mean())
    return pi1

def _make_row(label, v_true, v_false):
    """Build one stratification-table row from True/False norm arrays."""
    n_true = len(v_true)
    if n_true >= FREE_FIT_MIN_N:
        mu1_t, mu2_t, pi1_t, _ = _free_fit_pi1(v_true)
        fit_type = "free"
    else:
        mu1_t, mu2_t = full_mu1, full_mu2
        pi1_t = _constrained_pi1(v_true)
        fit_type = "constrained"
    # False-subset (always large; free fit). We only need pi1 for delta_pi1.
    if len(v_false) >= FREE_FIT_MIN_N:
        _, _, pi1_f, _ = _free_fit_pi1(v_false)
    else:
        pi1_f = _constrained_pi1(v_false)
    return {
        "Wordplay Type": label,
        "N (True)":      n_true,
        "Fit Type":      fit_type,
        "mu1":           mu1_t,
        "mu2":           mu2_t,
        "pi1 (True)":    pi1_t,
        "pi1 (False)":   pi1_f,
        "delta_pi1":     pi1_t - pi1_f,
    }

def _insert_rows_after(out, anchor_label, new_rows):
    """Insert a list of dict-rows into `out` directly after the row whose
    Wordplay Type equals `anchor_label`. If the anchor is missing, append
    to the end."""
    if not new_rows:
        return out
    anchor_pos = out.index[out["Wordplay Type"] == anchor_label]
    if len(anchor_pos):
        ins = int(anchor_pos[0]) + 1
        return pd.concat(
            [out.iloc[:ins], pd.DataFrame(new_rows), out.iloc[ins:]],
            ignore_index=True,
        )
    return pd.concat([out, pd.DataFrame(new_rows)], ignore_index=True)

def stratify_table(df_subset, label):
    """Build a wordplay-stratification summary table on a given subset.

    Layout: per-column wordplay rows sorted by |delta_pi1|, then four
    synthetic rows grouped together for side-by-side reading —
        double_def -> not double_def -> any letterplay -> no letterplay
    The synthetic block is placed by anchoring on double_def's sorted
    position and inserting the rest immediately after.
    """
    rows = []
    for col in wordplay_cols_kept:
        true_mask  = df_subset[col].astype(bool)
        v_true     = df_subset.loc[ true_mask, "fclue_norm_gstock"].to_numpy()
        v_false    = df_subset.loc[~true_mask, "fclue_norm_gstock"].to_numpy()
        if len(v_true) < MIN_TRUE:
            # Skipped earlier; safety fallback.
            continue
        rows.append(_make_row(col, v_true, v_false))
    out = pd.DataFrame(rows)
    out = out.reindex(out["delta_pi1"].abs().sort_values(ascending=False).index).reset_index(drop=True)

    # Synthetic complement of double_def. By construction its |delta_pi1|
    # equals double_def's (opposite sign), but its mu1, mu2, sigmas, and
    # absolute pi1 confirm that the two known modes persist with stable
    # locations and near-baseline weights when double_def clues are
    # excluded.
    synthetic_rows = []
    if "double_def" in df_subset.columns:
        dd_mask  = df_subset["double_def"].astype(bool)
        v_not_dd = df_subset.loc[~dd_mask, "fclue_norm_gstock"].to_numpy()
        v_dd     = df_subset.loc[ dd_mask, "fclue_norm_gstock"].to_numpy()
        if len(v_not_dd) >= MIN_TRUE:
            synthetic_rows.append(_make_row("not double_def", v_not_dd, v_dd))

        # Letterplay groupings restricted to non-double-def. "any letterplay"
        # captures clues with at least one detected letter-level device;
        # "no letterplay" captures clues with none. Together with double_def
        # they partition the full population, so the three subsets sum to
        # the total row count.
        present_lp = [c for c in LETTERPLAY_COLS if c in df_subset.columns]
        if present_lp:
            not_dd_mask = ~dd_mask
            any_lp_mask = df_subset[present_lp].astype(bool).any(axis=1)
            for syn_label, syn_true_mask in [
                ("any letterplay", not_dd_mask &  any_lp_mask),
                ("no letterplay",  not_dd_mask & ~any_lp_mask),
            ]:
                v_true_syn  = df_subset.loc[ syn_true_mask, "fclue_norm_gstock"].to_numpy()
                v_false_syn = df_subset.loc[~syn_true_mask, "fclue_norm_gstock"].to_numpy()
                if len(v_true_syn) >= MIN_TRUE:
                    synthetic_rows.append(_make_row(syn_label, v_true_syn, v_false_syn))

    # Anchor after double_def so the four synthetic rows appear adjacent
    # in the order: double_def, not double_def, any letterplay, no letterplay.
    out = _insert_rows_after(out, "double_def", synthetic_rows)
    return out

# All definitions, then start-only. Use the same `start` label produced by
# classify_position (verified in §3 cell 23).
wordplay_table_all   = stratify_table(fclue_full_df,                                   "all")
wordplay_table_start = stratify_table(fclue_full_df[fclue_full_df["def_position"] == "start"], "start")

print("=== All definitions: wordplay-type stratification (sorted by |delta_pi1|) ===")
with pd.option_context("display.float_format", "{:.4f}".format,
                       "display.width", 160, "display.max_colwidth", 30):
    print(wordplay_table_all.to_string(index=False))
print()
print("=== Start-definitions only: wordplay-type stratification (sorted by |delta_pi1|) ===")
with pd.option_context("display.float_format", "{:.4f}".format,
                       "display.width", 160, "display.max_colwidth", 30):
    print(wordplay_table_start.to_string(index=False))


In [ ]:
# === Figure: f_clue norms for selected wordplay types (fclue_bimodal_by_wordplay.png)
# One panel per type: density-normalized histogram of True-clues with a
# 2-component GMM overlay. The four canonical types (double_def, both
# anagram variants, hidden_fwd) are always shown; any additional type with
# |delta_pi1| > 0.03 in either table is added so the figure surfaces the
# largest mixture-weight shifts. The synthetic "not double_def" / "any
# letterplay" / "no letterplay" subsets are placed directly after
# double_def — in that order — when each crosses the threshold.
ALWAYS_SHOW_BASE = ["double_def", "anagram_single_word", "anagram_consec_words", "hidden_fwd"]
SYNTHETIC_AFTER_DD = ["not double_def", "any letterplay", "no letterplay"]
THRESHOLD          = 0.03

def _wp_mask(wp):
    """Boolean True-mask on fclue_full_df for a wordplay label, including
    the synthetic complement and letterplay groupings."""
    if wp == "not double_def":
        return ~fclue_full_df["double_def"].astype(bool)
    if wp in ("any letterplay", "no letterplay"):
        not_dd = ~fclue_full_df["double_def"].astype(bool)
        present_lp = [c for c in LETTERPLAY_COLS if c in fclue_full_df.columns]
        any_lp = fclue_full_df[present_lp].astype(bool).any(axis=1)
        return (not_dd & any_lp) if wp == "any letterplay" else (not_dd & ~any_lp)
    return fclue_full_df[wp].astype(bool)

# Union of always-show types and any type whose |delta_pi1| crosses the
# threshold in either the all-defs or start-defs table.
big_shift_types = set(
    wordplay_table_all  .loc[wordplay_table_all  ["delta_pi1"].abs() > THRESHOLD, "Wordplay Type"].tolist()
) | set(
    wordplay_table_start.loc[wordplay_table_start["delta_pi1"].abs() > THRESHOLD, "Wordplay Type"].tolist()
)

# Build the panel order: always-show types in canonical order. After the
# double_def panel, insert the synthetic complement and the letterplay
# groupings in canonical order, each conditional on the |delta_pi1|
# threshold. Threshold-driven additions for remaining real columns follow.
types_to_plot = []
for t in ALWAYS_SHOW_BASE:
    if t in wordplay_cols_kept:
        types_to_plot.append(t)
        if t == "double_def":
            for syn in SYNTHETIC_AFTER_DD:
                if syn in big_shift_types:
                    types_to_plot.append(syn)
known = set(ALWAYS_SHOW_BASE) | set(SYNTHETIC_AFTER_DD)
types_to_plot += [t for t in sorted(big_shift_types) if t not in known]
print(f"Plotting {len(types_to_plot)} wordplay types: {types_to_plot}")

# Match the binning convention used by figures 24/30/32 (start-defs range).
lo, hi = starts["fclue_norm_gstock"].min(), starts["fclue_norm_gstock"].max()
bin_edges = np.linspace(lo, hi, 61)

# Layout: up to 3 panels per row.
n_panels = len(types_to_plot)
n_cols = min(3, n_panels)
n_rows = int(np.ceil(n_panels / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4.5 * n_rows),
                         squeeze=False)
axes_flat = axes.ravel()

wordplay_fits = {}
for ax, wp in zip(axes_flat, types_to_plot):
    v = fclue_full_df.loc[_wp_mask(wp), "fclue_norm_gstock"].to_numpy()
    n_true = len(v)
    fit_type = "free" if n_true >= FREE_FIT_MIN_N else "constrained"

    # Warm contrasting color (#d62728) with diagonal hatch — distinguishes
    # the wordplay-stratified panels from the blue f_clue figures elsewhere.
    ax.hist(v, bins=bin_edges, alpha=0.45, color="#d62728",
            hatch="//", edgecolor="#d62728",
            density=True, label=f"True (N={n_true:,})")

    if fit_type == "free":
        fit = fit_gmm2(v)
    else:
        # Constrained: inherit means/sigmas from the full-dataset reference,
        # weights from posterior averaging on this subset.
        proba = full_gmm.predict_proba(v.reshape(-1, 1))
        order_full = np.argsort(full_gmm.means_.ravel())
        pi1 = float(proba[:, order_full[0]].mean())
        fit = {
            "mu1": full_mu1, "sigma1": float(full_sigmas[0]), "weight1": pi1,
            "mu2": full_mu2, "sigma2": float(full_sigmas[1]), "weight2": 1.0 - pi1,
        }
    wordplay_fits[wp] = fit
    overlay_gmm2(ax, v, fit, color="black", annotate_y_frac=0.92)

    ax.set_xlim(lo, hi)
    ax.set_xlabel("L2 norm")
    ax.set_ylabel("Density")
    ax.set_title(f"{wp}  ({fit_type} fit)")
    ax.grid(alpha=0.3)

    # Annotation block: N, pi1, pi2, fit type.
    ax.text(0.02, 0.78,
            f"N = {n_true:,}\npi1 = {fit['weight1']:.3f}\npi2 = {fit['weight2']:.3f}\n{fit_type}",
            transform=ax.transAxes, fontsize=8, va="top",
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.85))
    ax.legend(loc="upper right", fontsize=8)

# Hide any unused subplot axes when len(types_to_plot) < n_rows*n_cols.
for ax in axes_flat[n_panels:]:
    ax.set_axis_off()

fig.tight_layout()
fig.savefig(FIG_DIR / "fclue_bimodal_by_wordplay.png", dpi=300, bbox_inches="tight")
plt.show()

# Contribute every wordplay subset's GMM to the running summary table.
for wp, fit in wordplay_fits.items():
    gmm_fits.append({"figure": "fclue_bimodal_by_wordplay",
                     "distribution": f"g_stock f_clue norms (wordplay={wp}, True)",
                     **fit})


In [ ]:
# === Word identity: do the same definition words land in both modes?
per_def = starts.groupby("definition")["norm_group"].agg(
    n="size",
    n_lower=lambda s: (s == "lower").sum(),
    n_upper=lambda s: (s == "upper").sum(),
)
n_unique_defs = len(per_def)
n_in_both     = int(((per_def["n_lower"] > 0) & (per_def["n_upper"] > 0)).sum())
print(f"Unique start-def definition words: {n_unique_defs:,}")
print(f"  appearing in BOTH modes:         {n_in_both:,} "
      f"({n_in_both/n_unique_defs:.1%})")

# Within-word vs overall norm variability
overall_std = starts["fclue_norm_gstock"].std()
within_stds = starts.groupby("definition")["fclue_norm_gstock"].std().dropna()
within_mean_std = within_stds.mean()
print(f"Overall std of start-def norms:    {overall_std:.4f}")
print(f"Within-word mean std (unique defs with ≥2 clues): {within_mean_std:.4f}")
print(f"Within-word / overall ratio:       {within_mean_std/overall_std:.3f}")


In [ ]:
# === wndef norm correlation: does a definition's wndef norm predict its clue norm?
wndef_norms_gstock = np.linalg.norm(embeddings[("g_stock", "f_common_wndef")], axis=1)
starts = starts.assign(
    wndef_norm=wndef_norms_gstock[start_def_rows],
)
rho, _ = spearmanr(starts["wndef_norm"], starts["fclue_norm_gstock"])
pearson = starts[["wndef_norm", "fclue_norm_gstock"]].corr().iloc[0, 1]
print(f"Spearman rho(wndef_norm, fclue_norm) = {rho:.4f}")
print(f"Pearson  r  (wndef_norm, fclue_norm) = {pearson:.4f}")
print("(A small positive correlation would confirm the definition word itself "
      "contributes little to which mode the clue ends up in.)")


## §5 — Dimension-level characterization

Rather than looking for bimodal *individual* dimensions, we identify which
dimensions have systematically different mean values between the two norm
groups. This tells us which dimensions "assign" a clue to the lower or
upper mode — and whether the assignment is concentrated in a handful of
dimensions or distributed across many.


In [ ]:
# === Mean embeddings for lower- and upper-mode start-definitions
t0 = time.time()

start_emb_gstock = embeddings[("g_stock", "f_clue_val")][start_fclue_rows]
lower_mask = (starts["norm_group"] == "lower").to_numpy()
upper_mask = (starts["norm_group"] == "upper").to_numpy()

mean_lower = start_emb_gstock[lower_mask].mean(axis=0)
mean_upper = start_emb_gstock[upper_mask].mean(axis=0)
delta = mean_upper - mean_lower
assert delta.shape == (1024,)

pooled_var = (
    start_emb_gstock[lower_mask].var(axis=0) * (lower_mask.sum() - 1)
    + start_emb_gstock[upper_mask].var(axis=0) * (upper_mask.sum() - 1)
) / (lower_mask.sum() + upper_mask.sum() - 2)
pooled_std = np.sqrt(pooled_var)
cohens_d = delta / (pooled_std + 1e-10)

print(f"mean_lower shape: {mean_lower.shape}")
print(f"mean_upper shape: {mean_upper.shape}")
print(f"delta shape:      {delta.shape}")
print(f"Compute: {time.time() - t0:.1f}s")


In [ ]:
# === Figure: dimension-level delta profile (fclue_bimodal_dimensions.png)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Left: raw delta across all 1024 dimensions
axes[0].bar(np.arange(1024), delta, color=MODEL_COLORS["g_stock"], width=1.0)
axes[0].set_xlabel("Dimension index")
axes[0].set_ylabel("mean_upper − mean_lower")
axes[0].set_title("Per-dimension mean shift, upper vs lower norm mode (g_stock, start-defs)")
axes[0].grid(alpha=0.3)

# Right: sorted |delta|, top 50
top50 = np.argsort(np.abs(delta))[::-1][:50]
axes[1].bar(np.arange(50), np.abs(delta[top50]), color=MODEL_COLORS["g_stock"])
axes[1].set_xlabel("Rank by |delta|")
axes[1].set_ylabel("|mean_upper − mean_lower|")
axes[1].set_title("Top 50 dimensions by |delta|")
axes[1].grid(alpha=0.3)

fig.tight_layout()
fig.savefig(FIG_DIR / "fclue_bimodal_dimensions.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# === Top 10 dimensions by |delta|, with Cohen's d, plus concentration check
top10 = np.argsort(np.abs(delta))[::-1][:10]
top10_df = pd.DataFrame({
    "Dim":       top10,
    "delta":     delta[top10],
    "mean_lower":mean_lower[top10],
    "mean_upper":mean_upper[top10],
    "pooled_std":pooled_std[top10],
    "cohens_d":  cohens_d[top10],
})
with pd.option_context("display.float_format", "{:.4f}".format, "display.width", 140):
    print(top10_df.to_string(index=False))
print()

# Concentration: fraction of the (norm_upper^2 - norm_lower^2) difference
# explained by the top-k dimensions.
mean_norm_lower_sq = float((mean_lower ** 2).sum())
mean_norm_upper_sq = float((mean_upper ** 2).sum())
total_diff = mean_norm_upper_sq - mean_norm_lower_sq
print(f"Total (mean_upper^2 - mean_lower^2): {total_diff:.4f}")
print(f"(both means are computed on raw embeddings; above is ||mean_upper||^2 − ||mean_lower||^2)")
print()

conc_rows = []
for k in [10, 50, 100]:
    idx = np.argsort(np.abs(delta))[::-1][:k]
    partial = float((mean_upper[idx] ** 2).sum() - (mean_lower[idx] ** 2).sum())
    conc_rows.append({"k": k, "partial_diff": partial,
                      "fraction_of_total": partial / total_diff if total_diff != 0 else float("nan")})
conc_df = pd.DataFrame(conc_rows)
with pd.option_context("display.float_format", "{:.4f}".format, "display.width", 140):
    print("Concentration of the norm-squared difference by top-k dimensions:")
    print(conc_df.to_string(index=False))
print()

# Sanity check: is dim 379 the top-delta dimension?
rank_of_379 = int(np.where(np.argsort(np.abs(delta))[::-1] == 379)[0][0])
print(f"Sanity check — rank of dim 379 by |delta|: {rank_of_379} "
      f"(prior investigation found dim 379 had the highest norm correlation)")


## §6 — Does the bimodality propagate to cosine similarities?

L2 measures involving f_clue must inherit the norm bimodality; cosine
should not — unless the two norm groups also differ directionally. We
stratify T=1 cosine, T=1 L2, and ATE by norm group and check. We also
confirm that T=0 cosine irregularity is unrelated to f_clue norms.


In [ ]:
# === T=1 cosine, T=1 L2, and ATE stratified by norm group (g_stock, start-defs)
start_clue_emb = embeddings[("g_stock", "f_clue_val")][start_fclue_rows]
start_ans_emb  = embeddings[("g_stock", "f_common_wndef")][start_ans_rows]
start_def_emb  = embeddings[("g_stock", "f_common_wndef")][start_def_rows]

start_T1_cos = rowwise_cosine(start_clue_emb, start_ans_emb)
start_T1_l2  = np.linalg.norm(start_clue_emb - start_ans_emb, axis=1)
start_T0_cos = rowwise_cosine(start_def_emb, start_ans_emb)
start_T0_l2  = np.linalg.norm(start_def_emb - start_ans_emb, axis=1)

starts = starts.assign(
    T1_cos=start_T1_cos,
    T1_l2=start_T1_l2,
    T0_cos=start_T0_cos,
    T0_l2=start_T0_l2,
    ATE_cos=start_T1_cos - start_T0_cos,
)

strat_rows = []
for col in ["T1_cos", "T1_l2", "T0_cos", "ATE_cos"]:
    for grp in ["lower", "upper"]:
        v = starts.loc[starts["norm_group"] == grp, col].to_numpy()
        strat_rows.append({"Metric": col, "Group": grp,
                           "N": int(len(v)),
                           "Mean": float(v.mean()),
                           "Std":  float(v.std())})
strat_df = pd.DataFrame(strat_rows)
with pd.option_context("display.float_format", "{:.4f}".format, "display.width", 140):
    print(strat_df.to_string(index=False))


In [ ]:
# === Figure: stratified T=1 cosine and T=1 L2 (fclue_bimodal_propagation.png)
# Two panels (cosine, L2). Each panel has lower- and upper-norm-group
# overlays. A 2-component GMM is fit to each of the four distributions and
# overlaid in a darkened group color, so the reader can see whether the
# norm-group bimodality propagates to the similarity distributions.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Palette: distinct but within FIGURE_STANDARDS.md spirit — reuse g_stock
# color family with different shades for the two norm groups.
GROUP_COLORS = {"lower": "#aec7e8", "upper": "#1f77b4"}
GROUP_GMM_COLORS = {"lower": "#3a6ea5", "upper": "#0b3d91"}  # darkened

# Left: T=1 cosine
lo, hi = float(np.percentile(starts["T1_cos"],  0.5)), float(np.percentile(starts["T1_cos"], 99.5))
cos_bins = np.linspace(lo, hi, 61)
prop_cos_fits = {}
for grp in ["lower", "upper"]:
    v = starts.loc[starts["norm_group"] == grp, "T1_cos"].to_numpy()
    axes[0].hist(v, bins=cos_bins, alpha=0.55, color=GROUP_COLORS[grp],
                 label=f"{grp} (N={len(v):,}, mean={v.mean():.3f})",
                 density=True)
    axes[0].axvline(v.mean(), color=GROUP_COLORS[grp], linestyle="--", linewidth=1)
    fit = fit_gmm2(v)
    prop_cos_fits[grp] = fit
    overlay_gmm2(axes[0], v, fit, color=GROUP_GMM_COLORS[grp],
                 label_prefix=f"{grp} ",
                 annotate_y_frac=0.95 if grp == "lower" else 0.75)
axes[0].set_xlabel("T=1 cosine similarity")
axes[0].set_ylabel("Density")
axes[0].set_title("T=1 cosine by norm group (start-defs, g_stock)")
axes[0].grid(alpha=0.3)
axes[0].legend(loc="best", fontsize=8)

# Right: T=1 L2
lo, hi = float(np.percentile(starts["T1_l2"], 0.5)), float(np.percentile(starts["T1_l2"], 99.5))
l2_bins = np.linspace(lo, hi, 61)
prop_l2_fits = {}
for grp in ["lower", "upper"]:
    v = starts.loc[starts["norm_group"] == grp, "T1_l2"].to_numpy()
    axes[1].hist(v, bins=l2_bins, alpha=0.55, color=GROUP_COLORS[grp],
                 hatch="//", edgecolor=GROUP_COLORS[grp],
                 label=f"{grp} (N={len(v):,}, mean={v.mean():.3f})",
                 density=True)
    axes[1].axvline(v.mean(), color=GROUP_COLORS[grp], linestyle="--", linewidth=1)
    fit = fit_gmm2(v)
    prop_l2_fits[grp] = fit
    overlay_gmm2(axes[1], v, fit, color=GROUP_GMM_COLORS[grp],
                 label_prefix=f"{grp} ",
                 annotate_y_frac=0.95 if grp == "lower" else 0.75)
axes[1].set_xlabel("T=1 L2 distance")
axes[1].set_ylabel("Density")
axes[1].set_title("T=1 L2 by norm group (start-defs, g_stock)")
axes[1].grid(alpha=0.3)
axes[1].legend(loc="best", fontsize=8)

fig.tight_layout()
fig.savefig(FIG_DIR / "fclue_bimodal_propagation.png", dpi=300, bbox_inches="tight")
plt.show()

for grp, fit in prop_cos_fits.items():
    gmm_fits.append({"figure": "fclue_bimodal_propagation",
                     "distribution": f"T=1 cosine ({grp} norm group)",
                     **fit})
for grp, fit in prop_l2_fits.items():
    gmm_fits.append({"figure": "fclue_bimodal_propagation",
                     "distribution": f"T=1 L2 ({grp} norm group)",
                     **fit})


In [ ]:
# === T=0 cosine is independent of f_clue norm (expected)
# T=0 uses only wndef embeddings, so there should be no correlation between
# a clue's f_clue norm and its T=0 cosine.
rho_t0, _ = spearmanr(starts["fclue_norm_gstock"], starts["T0_cos"])
rho_t1, _ = spearmanr(starts["fclue_norm_gstock"], starts["T1_cos"])
rho_ate, _ = spearmanr(starts["fclue_norm_gstock"], starts["ATE_cos"])
print(f"Spearman rho(f_clue norm, T=0 cosine):   {rho_t0:.4f}")
print(f"Spearman rho(f_clue norm, T=1 cosine):   {rho_t1:.4f}")
print(f"Spearman rho(f_clue norm, ATE cosine):   {rho_ate:.4f}")


## §7 — Effect of fine-tuning

We briefly confirm that g1 eliminates the bimodality as a side effect of
global compression. The primary exhibit is the g_stock vs g1 overlay with
bimodal vs unimodal structure annotated; we also run the §3 position split
under g1 and expect both positions to be unimodal.


In [ ]:
# === g1 f_clue norms by definition position (expected: both unimodal)
# Even though both distributions are expected to be unimodal under g1, we
# fit a 2-component GMM to confirm: if g1 has truly collapsed the modes,
# the two GMM components should overlap heavily (similar μ values) or one
# should dominate the mixture weight.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

lo, hi = fclue_norms_g1.min(), fclue_norms_g1.max()
bin_edges = np.linspace(lo, hi, 61)

g1_position_fits = {}
for ax, which in zip(axes, ["start", "end"]):
    v = clues_val_kept.loc[clues_val_kept["def_position"] == which, "fclue_norm_g1"].to_numpy()
    ax.hist(v, bins=bin_edges, alpha=0.55, color=MODEL_COLORS["g1"],
            hatch="//", edgecolor=MODEL_COLORS["g1"],
            density=True, label=f"{which} (N={len(v):,}, std={v.std():.3f})")
    ax.axvline(v.mean(), color=MODEL_COLORS["g1"], linestyle="--", linewidth=1)
    ax.set_xlabel("L2 norm")
    ax.set_ylabel("Density")
    ax.set_title(f"g1 f_clue norms — {which}-of-surface definitions")
    ax.grid(alpha=0.3)

    fit = fit_gmm2(v)
    g1_position_fits[which] = fit
    overlay_gmm2(ax, v, fit, color="#a04000", annotate_y_frac=0.92)
    ax.legend(loc="best", fontsize=8)

fig.tight_layout()
fig.savefig(FIG_DIR / "fclue_bimodal_g1_by_position.png", dpi=300, bbox_inches="tight")
plt.show()

for which, fit in g1_position_fits.items():
    gmm_fits.append({"figure": "fclue_bimodal_g1_by_position",
                     "distribution": f"g1 f_clue norms ({which}-defs)",
                     **fit})

# Summary stats comparison
ft_rows = []
for model, col in [("g_stock", "fclue_norm_gstock"), ("g1", "fclue_norm_g1")]:
    for which in ["start", "end"]:
        v = clues_val_kept.loc[clues_val_kept["def_position"] == which, col].to_numpy()
        ft_rows.append({"Model": model, "Position": which,
                        "N":    int(len(v)),
                        "Mean": float(v.mean()),
                        "Std":  float(v.std())})
ft_df = pd.DataFrame(ft_rows)
with pd.option_context("display.float_format", "{:.4f}".format, "display.width", 140):
    print(ft_df.to_string(index=False))


In [ ]:
# === GMM fit summary table: all distributions
# Build a single DataFrame from all GMM fits collected by the figure cells
# above. The "peak locations" view focuses on (mu1, mu2, weight1) so the
# reader can scan for recurring peak values across distributions — if μ1
# and μ2 cluster around the same two values regardless of which f, which
# definition position, or which similarity metric we look at, that is
# evidence for natural modes in CALE's embedding space rather than
# feature-specific bimodalities.
gmm_summary_df = pd.DataFrame(gmm_fits)
gmm_summary_df = gmm_summary_df[[
    "figure", "distribution",
    "mu1", "sigma1", "weight1",
    "mu2", "sigma2", "weight2",
]]

print("=== Full GMM fit summary ===")
with pd.option_context("display.float_format", "{:.4f}".format,
                       "display.width", 200,
                       "display.max_colwidth", 60,
                       "display.max_rows", None):
    print(gmm_summary_df.to_string(index=False))

print()
print("=== Peak locations (compact) ===")
peaks_df = gmm_summary_df[["figure", "distribution", "mu1", "mu2", "weight1"]]
with pd.option_context("display.float_format", "{:.4f}".format,
                       "display.width", 200,
                       "display.max_colwidth", 60,
                       "display.max_rows", None):
    print(peaks_df.to_string(index=False))


## §8 — Summary

This section is populated by the numerical results above; the findings
listed here summarize what the run produced.

**Findings:**

- **The bimodality is real, not a computational error.** §2 confirmed the
  same two-peak structure appears in the full dataset and the validation
  slice. Both the norm histogram and the T=1 L2 distribution inherit it;
  cosine does not.
- **Phrase format changes direction, not just magnitude.** The mean cosine
  between f_wndef(word) and f_wnex(word) for the same 8,360 words is
  0.720 under g_stock — the two phrase formats produce
  embeddings that point in meaningfully different directions, not just at
  different magnitudes. Under g1, the alignment decreases to
  0.633.
- **Definition position modulates which norm mode dominates, not whether
  bimodality exists.** §3 showed that both definition positions exhibit
  two norm modes at the same locations (~29.5 and ~31.5); what changes is
  the balance. Start-definitions preferentially land in the upper mode;
  end-definitions preferentially land in the lower mode. The overall
  distribution looks bimodal because start-defs outnumber end-defs, and
  the upper mode gets the extra weight.
- **No observable text property explains mode assignment among
  start-definitions.** §4 ruled out word count, definition character
  length, subword token count, publisher/source, and definition word
  identity. Within-word norm variability is comparable to overall
  variability, confirming context — not the word itself — drives which
  mode a clue lands in.
- **No single wordplay device drives the bimodality.** §4 stratified the
  full 239,406-row f_clue dataset by algorithmically detected wordplay type
  (anagrams, hidden words, double definitions, letter-selection patterns).
  Mode locations stayed within ~0.1 of the reference (mu1, mu2) across every
  subset; what differs is the mixture weight pi1, with the largest shifts
  appearing for the wordplay types reported in the §4 tables. No subset
  collapses onto a single mode, confirming the bimodality is not produced
  by any one structural pattern.
- **It is a distributed, not sparse, model behavior.** §5 showed the
  upper-vs-lower mode shift is spread across many embedding dimensions.
  The concentration-check fractions reported above quantify how much of
  the norm-squared difference is captured by the top 10/50/100 dimensions.
- **Propagation to cosine and ATE is limited.** §6 showed T=1 cosine and
  ATE differ only modestly between norm groups, while T=1 L2 differs
  substantially (as expected). T=0 cosine is uncorrelated with f_clue
  norm, confirming the two irregular distributions are independent
  phenomena.
- **g1 eliminates the bimodality.** §7 confirmed that fine-tuning's global
  norm compression collapses both modes into a single tight distribution,
  regardless of definition position.

- **GMM fits reveal strikingly consistent peak locations across every g_stock f_clue distribution.** A 2-component GMM was fit to every histogram in this notebook. The g_stock f_clue norm peaks recur at **(μ₁≈29.6, μ₂≈31.6)** for the full dataset, the validation slice, start-defs, end-defs, 1-token defs, 2-token defs, and all three top publishers — varying by less than 0.15 across all 11 slicings. What changes is the *mixture weight*: end-defs have weight₁=0.59 (lower mode dominant) while start-defs have weight₁=0.43 (upper mode dominant), confirming §3's finding that definition position modulates *which* mode dominates rather than *where* the modes sit. The uncontextualized vocabulary distributions (f_wndef, f_wnex) sit ≈1–2 units lower (μ₁≈27.8–29.4, μ₂≈29.9–31.0) and have noticeably wider components (σ≈1.0 vs. σ≈0.7 for f_clue), reinforcing that the tight bimodality is a clue-context phenomenon. Under g1, both f_clue norm components collapse onto each other (μ₁≈28.4, μ₂≈29.4, Δμ≈0.9 vs. ≈2.0 for g_stock), consistent with §7's qualitative finding that fine-tuning eliminates the bimodality.

**Implication.** L2-based analyses involving g_stock f_clue embeddings
should be interpreted with the bimodality in mind. Cosine-based analyses
are largely unaffected. The ATE under g_stock is minimally confounded.
Under g1, the bimodality does not appear, so this specific concern does
not carry forward to fine-tuned evaluations.

In [ ]:
# === Build outputs/cale_fclue_norm_bimodality-results.md
t0 = time.time()
RESULTS_PATH = OUTPUT_DIR / "results" / "cale_fclue_norm_bimodality-results.md"

def df_md(df, fmt=".4f"):
    return df.to_markdown(index=False, floatfmt=fmt)

lines = []
lines.append("# CALE f_clue Norm Bimodality Investigation — Results")
lines.append("")
lines.append(f"Generated: {date.today().isoformat()}")
lines.append("")
lines.append("## Versions")
lines.append("")
for k, v in VERSIONS.items():
    lines.append(f"- **{k}:** {v}")
lines.append("")

lines.append("## §2 — Full dataset vs validation slice norms")
lines.append("")
lines.append(df_md(norm_fullval_df, fmt=".4f"))
lines.append("")

lines.append("## §2 — g_stock vocabulary norms (contrast to f_clue)")
lines.append("")
lines.append(df_md(vocab_norm_df, fmt=".4f"))
lines.append("")
lines.append("Both vocabulary distributions are visibly unimodal, "
             "confirming the bimodality is specific to the clue-context setting.")
lines.append("")
lines.append("## §2 — Cross-format directional alignment (wndef vs wnex, same words)")
lines.append("")
lines.append(df_md(cross_cos_df, fmt=".4f"))
lines.append("")
lines.append("## §2 — T=0 / T=1 distribution summary (both models)")
lines.append("")
lines.append(df_md(t01_df, fmt=".4f"))
lines.append("")

lines.append("## §3 — Norm stats by definition position (g_stock)")
lines.append("")
lines.append(df_md(pos_stats_df, fmt=".4f"))
lines.append("")

lines.append("## §4 — Top publishers by start-def count")
lines.append("")
lines.append(df_md(pub_stats.head(10).reset_index(), fmt=".4f"))
lines.append("")
lines.append(f"- Unique start-def definition words: **{n_unique_defs:,}**")
lines.append(f"- Appearing in both modes: **{n_in_both:,} "
             f"({n_in_both/n_unique_defs:.1%})**")
lines.append(f"- Within-word mean std / overall std: **{within_mean_std/overall_std:.3f}**")
lines.append(f"- Spearman rho(wndef_norm, fclue_norm): **{rho:.4f}**")
lines.append("")

lines.append("## §4 — Wordplay type stratification (all definitions)")
lines.append("")
lines.append(df_md(wordplay_table_all, fmt=".4f"))
lines.append("")
lines.append("## §4 — Wordplay type stratification (start-definitions only)")
lines.append("")
lines.append(df_md(wordplay_table_start, fmt=".4f"))
lines.append("")
lines.append("## §5 — Top 10 dimensions by |delta|")
lines.append("")
lines.append(df_md(top10_df, fmt=".4f"))
lines.append("")
lines.append("### Concentration of norm-squared difference by top-k dims")
lines.append("")
lines.append(df_md(conc_df, fmt=".4f"))
lines.append("")
lines.append(f"- Rank of dim 379 by |delta|: **{rank_of_379}**")
lines.append("")

lines.append("## §6 — T=0 / T=1 / ATE stratified by norm group (start-defs)")
lines.append("")
lines.append(df_md(strat_df, fmt=".4f"))
lines.append("")
lines.append(f"- Spearman rho(f_clue norm, T=0 cosine):   **{rho_t0:.4f}**")
lines.append(f"- Spearman rho(f_clue norm, T=1 cosine):   **{rho_t1:.4f}**")
lines.append(f"- Spearman rho(f_clue norm, ATE cosine):   **{rho_ate:.4f}**")
lines.append("")

lines.append("## §7 — g_stock vs g1 f_clue norms by definition position")
lines.append("")
lines.append(df_md(ft_df, fmt=".4f"))
lines.append("")

lines.append("## GMM fit summary (all distributions)")
lines.append("")
lines.append(df_md(gmm_summary_df, fmt=".4f"))
lines.append("")

lines.append("## Figures")
lines.append("")
for fig_name in [
    "fclue_bimodal_norm_full_vs_val.png",
    "fclue_bimodal_vocab_norms.png",
    "fclue_bimodal_cross_format_cosine.png",
    "fclue_bimodal_norm_overlay.png",
    "fclue_bimodal_t0_t1.png",
    "fclue_bimodal_by_position.png",
    "fclue_bimodal_by_subword_tokens.png",
    "fclue_bimodal_by_source.png",
    "fclue_bimodal_by_wordplay.png",
    "fclue_bimodal_dimensions.png",
    "fclue_bimodal_propagation.png",
    "fclue_bimodal_g1_by_position.png",
]:
    lines.append(f"- `figures/{fig_name}`")
lines.append("")

RESULTS_PATH.write_text("\n".join(lines))
print(f"Wrote {RESULTS_PATH} ({RESULTS_PATH.stat().st_size:,} bytes) "
      f"in {time.time() - t0:.1f}s")


In [ ]:
# === Wall-clock runtime
elapsed = time.time() - NOTEBOOK_T0
print(f"Notebook wall-clock runtime: {elapsed:.1f}s ({elapsed/60:.1f} min)")
